# QRoadScan Gemma 4 E2B LiteRT-LM T4 Hive Blog Forge — V3 Mesh

This notebook upgrades the QRoadScan blog writer from a single local Gemma session into a configurable **1-10 worker Gemma hive**. Each worker owns its own LiteRT-LM engine and cache, while a shared in-memory intercom surface lets workers exchange plans, citation warnings, draft fingerprints, style moves, and risk notes while the batch is being written.

What changed in V3:

- `GEMMA_THREAD_COUNT` lets you run 1-10 concurrent Gemma workers.
- `AUTO_SCALE_THREADS_TO_T4` reads `nvidia-smi` and caps the worker count against available T4 VRAM using a configurable per-worker estimate.
- Colab GPU is now strict by default: `INFERENCE_BACKEND = "GPU"`, `ALLOW_CPU_FALLBACK = False`, and the notebook stops if LiteRT cannot create a GPU engine.
- Model storage is disk-sane by default: `MODEL_STORAGE_MODE = "plain"` keeps one verified `.litertlm` runtime file instead of encrypted + decrypted + cache duplicates.
- AES-GCM encryption is still available with `MODEL_STORAGE_MODE = "encrypted"`, but it is no longer the default for Colab GPU runs because LiteRT requires a real file path.
- Colab Secrets are first-class when encryption is enabled: `QRS_MODEL_VAULT_PASSPHRASE` is read from `google.colab.userdata`, hydrated into `os.environ`, and audited without exposing the value.
- Optional Hugging Face token secrets (`HF_TOKEN`, `HUGGINGFACE_TOKEN`, or `HUGGING_FACE_HUB_TOKEN`) are used for authenticated model downloads.
- Every worker launches its own LiteRT-LM engine, cache, persona, and generation loop.
- A thread-safe `HiveIntercom` builds a large live surface between workers so each article can learn from other articles without blocking the whole run.
- The prompt kernel now uses planner, bridge, draft, fact-guard, and editor contracts with stricter citation discipline and stronger human-writing moves.
- A background telemetry sampler records real GPU pressure, and a local article quality audit records citation count, duplicate paragraphs, phrase warnings, and a quality score.
- The scheduler writes the final Markdown bundle in article order even when workers finish out of order.

Run order:

1. Install pinned dependencies.
2. Add a Colab Secret named `QRS_MODEL_VAULT_PASSPHRASE` in the left sidebar key panel, or keep interactive prompting enabled.
3. Edit the single control cell, especially `ARTICLE_COUNT` and `GEMMA_THREAD_COUNT`.
4. Download / verify / seal Gemma 4 E2B.
5. Load LiteRT-LM runtime and local RAG surfaces.
6. Run the T4 hive batch generator.

Notes:

- The model remains encrypted at rest with AES-GCM. During threaded inference, the notebook decrypts one temporary plaintext model inside a context manager, loads independent engines from it, then deletes it when the hive exits.
- Local RAG citations remain deterministic and use compact tokens such as `⟦SFC:surface#chunk@hash⟧`.
- The quantum/RGB simulation is private routing and style telemetry only. It is not treated as factual road evidence.


In [ ]:
# Cell 0 — Pinned Colab installs
# Runtime > Restart runtime after this cell if Colab asks.

!pip -q install   litert-lm==0.10.2   httpx==0.28.0   cryptography==46.0.1   psutil==6.1.1   pennylane==0.41.0   bleach==6.3.0   tqdm==4.66.5   accelerate   transformers   huggingface_hub   safetensors   sentencepiece   bitsandbytes

print("Dependencies installed. If Colab asks, restart runtime, then continue at Cell 1.")


In [ ]:
# Cell 1 — SINGLE CONTROL CELL: topics, hive size, brand facts, generation mode
# Edit only this cell for normal use.

from pathlib import Path
import os, textwrap

# Generate 1 to 42 Markdown posts in one run.
ARTICLE_COUNT = 10  # <-- set 1..42

# Run 1 to 10 independent Gemma worker threads.
# On a T4, start with 2-4 for real GPU engines, then raise it if VRAM remains free.
GEMMA_THREAD_COUNT = 3  # <-- set 1..10
AUTO_SCALE_THREADS_TO_T4 = True
T4_TARGET_VRAM_UTILIZATION = 0.88
APPROX_VRAM_GB_PER_WORKER = 2.20
MIN_FREE_VRAM_GB_AFTER_LOAD = 1.80
THREAD_STARTUP_STAGGER_SECONDS = 2.50

# Hive/intercom knobs.
HIVE_MODE = "mesh"  # solo, queue, mesh. mesh uses the live inter-worker surface.
INTERCOM_ROUNDS = 1  # 0..3 extra bridge notes per article. Higher = richer but slower.
INTERCOM_MAX_SURFACE_CHARS = 16000
INTERCOM_PACKET_CHARS = 2400
INTERCOM_MEMORY_ITEMS = 140
ENABLE_SUMMA_COMPACTION = True
RAG_TOP_K = 12

# Add as many topics as you want. If ARTICLE_COUNT exceeds this list, topic variants are auto-expanded.
BLOG_TOPICS = [
    "How QRoadScan turns traffic risk signals into calmer route decisions",
    "Why a live road risk colorwheel can help drivers understand hazards faster",
    "Road hazard alerts, confidence scoring, and the future of safer commutes",
    "AI-assisted traffic awareness without overwhelming the driver",
    "From route scans to saved reports: what a road safety dashboard should explain",
    "How predictive safety context can reduce uncertainty before a trip",
]

BRAND_NAME = "QRoadScan.com"
BRAND_URL = "https://qroadscan.com"
BRAND_VOICE = "human, practical, safety-aware, technical but readable, quietly futuristic"
TARGET_READER = "drivers, road-safety researchers, fleet operators, traffic-tech builders, and safety-conscious commuters"

BRAND_CONTEXT = """
QRoadScan.com is a road intelligence and safety awareness product focused on live traffic risk visualization, route scans, road hazard alerts, saved reports, and AI-assisted driving safety insights.
The product concept compresses changing route signals into a readable risk pulse: one reading, one confidence score, practical reasons, and driver-ready next steps.
Core motifs: live risk map, road hazard awareness, safer route planning, colorwheel feedback, route scan reports, confidence scoring, calmer decisions, dashboard intelligence, and real-world driving context.
""".strip()

# Local RAG surfaces: add factual notes, product docs, landing page copy, research summaries, or quotes you own.
# Each item can have title, url, and body. The writer cites these with local surface citations.
EXTRA_RAG_SURFACES = [
    {
        "title": "QRoadScan product positioning",
        "url": BRAND_URL,
        "body": BRAND_CONTEXT,
    },
    {
        "title": "QRoadScan blog direction",
        "url": f"{BRAND_URL}/blog",
        "body": "Short reads about road intelligence, safer routes, traffic risk, road hazards, and product updates.",
    },
]

# Optional folder for extra .txt, .md, .json, .csv files you upload into Colab.
# Example: upload docs to /content/qroadscan_sources before running the batch cell.
SOURCE_FOLDER = Path("/content/qroadscan_sources")
OUTPUT_DIR = Path("/content/qroadscan_blog_markdowns_v3_hive")

# Agent depth: "fast" = planner + draft, "deep" = guard + editor, "hive-deep" = deep plus intercom bridge.
AGENT_DEPTH = "hive-deep"

# Style knobs.
WORDS_PER_ARTICLE = 1200
TEMPERATURE_STYLE = "balanced-human"  # options: crisp, balanced-human, cinematic, founder-note, field-notes
INCLUDE_TECHNICAL_APPENDIX = True
INCLUDE_MEDIUM_READY_FRONTMATTER = True
INCLUDE_LOCAL_CITATION_LEDGER = True

# Worker personas rotate across the hive so tiny models behave more like a coordinated editorial desk.
WORKER_PERSONAS = [
    "route-safety systems writer: concrete, calm, operational",
    "product narrative strategist: crisp framing, founder-level clarity",
    "driver empathy editor: practical scenes, low hype, human rhythm",
    "citation sentinel: cautious claims, surface-first evidence discipline",
    "dashboard UX explainer: turns telemetry into readable decisions",
    "future-of-mobility essayist: speculative only when clearly labeled",
    "fleet operations analyst: repeatable workflows and risk communication",
    "technical translator: plain-language systems explanations",
    "Medium polish editor: strong hooks, clean sections, no filler",
    "intercom synthesizer: finds patterns across the hive memory",
]

# Model runtime knobs.
# Colab default is CUDA Transformers because LiteRT-LM GPU can expose Backend.GPU yet fail
# inside the compiled model executor on T4 before VRAM allocation. LiteRT paths remain available.
MODEL_ENGINE = "transformers-cuda"  # transformers-cuda, auto, litert-gpu, litert-cpu
LITERT_GPU_FALLBACK_ENGINE = "transformers-cuda"
INFERENCE_BACKEND = "GPU"  # Used by LiteRT engines only: GPU, Auto, or CPU.
ALLOW_CPU_FALLBACK = False
REQUIRE_GPU_RUNTIME = True
MAX_PROMPT_CHARS = 28000
MAX_ARTICLE_CHARS = 32000
ENABLE_ENGINE_WARMUP = True  # Makes GPU allocation visible before the batch starts.
WARMUP_PROMPT = "Return exactly: hive worker ready"

# CUDA Transformers runtime: uses the real T4 through torch/accelerate.
TRANSFORMERS_MODEL_ID = "google/gemma-4-E2B-it"
TRANSFORMERS_CACHE_DIR = "/content/qroadscan_gemma4_e2b_blog_forge/hf_cache"
TRANSFORMERS_LOAD_IN_4BIT = False  # False = fp16, fills more T4 VRAM. True = lower VRAM.
TRANSFORMERS_DEVICE_MAP = "cuda"  # cuda forces all weights onto GPU; auto may offload.
TRANSFORMERS_MAX_NEW_TOKENS = 1800
TRANSFORMERS_TEMPERATURE = 0.72
TRANSFORMERS_TOP_P = 0.92
MAX_TRANSFORMERS_CONCURRENT_GENERATIONS = 1
TRANSFORMERS_TRUST_REMOTE_CODE = True

# Disk-sane LiteRT storage. Skipped entirely when MODEL_ENGINE="transformers-cuda".
PURGE_LITERT_ARTIFACTS_WHEN_TRANSFORMERS = True
MODEL_STORAGE_MODE = "plain"  # plain or encrypted
MODEL_VERIFY_EACH_RUN = False  # False trusts prior verified model metadata to avoid huge re-hashes.
CLEANUP_STALE_MODEL_ARTIFACTS = True
PURGE_ENCRYPTED_MODEL_WHEN_PLAIN_MODE = True
ENABLE_LITERT_CACHE = False  # Avoid per-worker cache directories unless you explicitly want them.
DELETE_PLAINTEXT_AFTER_ENCRYPT = True
ALLOW_DECRYPT_ENCRYPTED_TO_PLAIN_ON_MODE_SWITCH = False  # Keep false if the old encrypted artifact is suspect.

# Hive telemetry and local quality audit knobs.
ENABLE_HIVE_TELEMETRY = True
HIVE_TELEMETRY_INTERVAL_SECONDS = 4.0
ENABLE_LOCAL_QUALITY_AUDIT = True
QUALITY_MIN_CITATIONS = 1
QUALITY_WARN_PHRASES = [
    "in today's fast-paced world",
    "game changer",
    "revolutionary",
    "seamlessly",
    "unlock the power",
]

# Security knobs. Preferred path: Colab Secrets, left sidebar key icon.
# Add QRS_MODEL_VAULT_PASSPHRASE as a Colab Secret and grant notebook access.
MODEL_VAULT_ENV = "QRS_MODEL_VAULT_PASSPHRASE"
MODEL_VAULT_SECRET_NAMES = [
    "QRS_MODEL_VAULT_PASSPHRASE",
    "MODEL_VAULT_PASSPHRASE",
    "GEMMA_MODEL_VAULT_PASSPHRASE",
]
REQUIRE_SECRET_FOR_VAULT = False  # True disables interactive passphrase fallback.
EXPORT_COLAB_SECRETS_TO_ENV = True

# Optional authenticated Hugging Face download support. Public models do not need this,
# but gated/private mirrors can use any one of these Colab Secret names.
HF_TOKEN_SECRET_NAMES = ["HF_TOKEN", "HUGGINGFACE_TOKEN", "HUGGING_FACE_HUB_TOKEN"]

print(f"Configured {ARTICLE_COUNT} article(s) for {BRAND_NAME}.")
print(f"Requested Gemma workers: {GEMMA_THREAD_COUNT} (auto-scale={AUTO_SCALE_THREADS_TO_T4})")
print(f"Model engine: {MODEL_ENGINE}; LiteRT backend={INFERENCE_BACKEND} (allow CPU fallback={ALLOW_CPU_FALLBACK})")
print(f"Transformers model: {TRANSFORMERS_MODEL_ID}; 4bit={TRANSFORMERS_LOAD_IN_4BIT}; device_map={TRANSFORMERS_DEVICE_MAP}")
print(f"LiteRT model storage mode: {MODEL_STORAGE_MODE}; LiteRT cache enabled={ENABLE_LITERT_CACHE}")
print(f"Output folder: {OUTPUT_DIR}")


In [ ]:
# Cell 2 — Download, SHA-256 verify, AES-GCM seal Gemma 4 E2B LiteRT-LM
# Inspired by the model validation and encrypted model handling pattern in your app.

from __future__ import annotations

import base64
import contextlib
import hashlib
import json
import os
import secrets
import shutil
import struct
import sys
import tempfile
import time
import uuid
from dataclasses import dataclass
from getpass import getpass
from pathlib import Path
from typing import Any, Callable, Dict, Iterable, Iterator, List, Optional, Tuple

import httpx
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
from cryptography.hazmat.primitives.kdf.scrypt import Scrypt

MODEL_REPO = "https://huggingface.co/litert-community/gemma-4-E2B-it-litert-lm/resolve/main/"
MODEL_FILE = "gemma-4-E2B-it.litertlm"
EXPECTED_HASH = "ab7838cdfc8f77e54d8ca45eadceb20452d9f01e4bfade03e5dce27911b27e42"
NETWORK_TIMEOUT = httpx.Timeout(connect=15.0, read=120.0, write=120.0, pool=15.0)

ROOT_DIR = Path("/content/qroadscan_gemma4_e2b_blog_forge")
MODELS_DIR = ROOT_DIR / "models"
CACHE_DIR = ROOT_DIR / ".litert_cache"
TEMP_DIR = ROOT_DIR / ".tmp"
LEDGER_DIR = ROOT_DIR / "ledgers"
PLAIN_MODEL_PATH = MODELS_DIR / MODEL_FILE
ENCRYPTED_MODEL_PATH = MODELS_DIR / f"{MODEL_FILE}.aesgcm"
MODEL_LEDGER_PATH = LEDGER_DIR / "model_integrity_ledger.jsonl"

FILE_ENCRYPTION_MAGIC = b"QRSBLOG2"
FILE_ENCRYPTION_CHUNK_SIZE = 4 * 1024 * 1024

for d in (ROOT_DIR, MODELS_DIR, CACHE_DIR, TEMP_DIR, LEDGER_DIR, OUTPUT_DIR, SOURCE_FOLDER):
    d.mkdir(parents=True, exist_ok=True)
    try:
        os.chmod(d, 0o700)
    except Exception:
        pass


def _chmod_private(path: Path, is_dir: bool = False) -> None:
    try:
        os.chmod(path, 0o700 if is_dir else 0o600)
    except Exception:
        pass


def sanitize_text(value: Any, *, max_chars: int = 4000) -> str:
    import re
    text = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]", "", str(value or "")).strip()
    return text[:max_chars].rstrip() if len(text) > max_chars else text


def human_size(value: int) -> str:
    units = ["B", "KB", "MB", "GB", "TB"]
    size = float(max(0, value))
    unit = units[0]
    for unit in units:
        if size < 1024 or unit == units[-1]:
            break
        size /= 1024.0
    return f"{size:.1f}{unit}" if unit != "B" else f"{int(size)}B"


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def derive_password_key(passphrase: str, salt: bytes) -> bytes:
    # Colab-friendly Scrypt settings. Increase n for slower, stronger offline protection.
    kdf = Scrypt(salt=salt, length=32, n=2**14, r=8, p=1)
    return kdf.derive(passphrase.encode("utf-8"))


SECRET_RUNTIME_AUDIT: Dict[str, str] = {}
COLAB_SECRET_AUDIT: Dict[str, Dict[str, str]] = {}


def _unique_secret_names(names: Iterable[str]) -> List[str]:
    out: List[str] = []
    for name in names:
        clean = sanitize_text(name, max_chars=120)
        if clean and clean not in out:
            out.append(clean)
    return out


def _read_colab_secret(name: str) -> str:
    try:
        from google.colab import userdata  # type: ignore
    except Exception as exc:
        COLAB_SECRET_AUDIT[name] = {"provider": "colab", "status": f"not_available:{type(exc).__name__}"}
        return ""
    try:
        value = userdata.get(name)
    except Exception as exc:
        COLAB_SECRET_AUDIT[name] = {"provider": "colab", "status": f"unreadable:{type(exc).__name__}"}
        return ""
    value = str(value or "").strip()
    if not value:
        COLAB_SECRET_AUDIT[name] = {"provider": "colab", "status": "missing_or_empty"}
        return ""
    COLAB_SECRET_AUDIT[name] = {"provider": "colab", "status": "loaded"}
    return value


def secret_value(primary_name: str, aliases: Optional[Iterable[str]] = None, *, label: str = "secret") -> Tuple[str, str]:
    names = _unique_secret_names([primary_name] + list(aliases or []))
    export_to_env = bool(globals().get("EXPORT_COLAB_SECRETS_TO_ENV", True))

    for name in names:
        value = os.environ.get(name, "").strip()
        if value:
            SECRET_RUNTIME_AUDIT[label] = f"env:{name}"
            if export_to_env:
                os.environ.setdefault(primary_name, value)
            return value, f"env:{name}"

    for name in names:
        value = _read_colab_secret(name)
        if value:
            if export_to_env:
                os.environ.setdefault(primary_name, value)
                os.environ.setdefault(name, value)
            SECRET_RUNTIME_AUDIT[label] = f"colab_secret:{name}"
            return value, f"colab_secret:{name}"

    SECRET_RUNTIME_AUDIT[label] = "not_found"
    return "", "not_found"


def model_vault_passphrase() -> str:
    aliases = globals().get("MODEL_VAULT_SECRET_NAMES", [])
    passphrase, source = secret_value(MODEL_VAULT_ENV, aliases, label="model_vault_passphrase")
    if not passphrase:
        if bool(globals().get("REQUIRE_SECRET_FOR_VAULT", False)):
            raise ValueError(
                "Model vault passphrase was not found in environment variables or Colab Secrets. "
                f"Add a Colab Secret named {MODEL_VAULT_ENV}, or set REQUIRE_SECRET_FOR_VAULT=False."
            )
        passphrase = getpass(f"Enter model vault passphrase for {MODEL_VAULT_ENV}: ").strip()
        source = "interactive_prompt"
        SECRET_RUNTIME_AUDIT["model_vault_passphrase"] = source
    if len(passphrase) < 12:
        raise ValueError("Use a model vault passphrase of at least 12 characters.")
    print(f"Model vault passphrase loaded from {source}.")
    return passphrase


def huggingface_token() -> Tuple[str, str]:
    names = _unique_secret_names(globals().get("HF_TOKEN_SECRET_NAMES", ["HF_TOKEN"]))
    if not names:
        return "", "not_configured"
    return secret_value(names[0], names[1:], label="huggingface_token")


def model_download_headers() -> Dict[str, str]:
    token, source = huggingface_token()
    if token:
        print(f"Hugging Face token loaded from {source}; using authenticated model download.")
        return {"Authorization": f"Bearer {token}"}
    return {}


def vault_key_from_passphrase(passphrase: str) -> bytes:
    salt_path = ROOT_DIR / ".model_vault_salt"
    if salt_path.exists():
        salt = salt_path.read_bytes()
    else:
        salt = secrets.token_bytes(16)
        salt_path.write_bytes(salt)
        _chmod_private(salt_path)
    return derive_password_key(passphrase, salt)


def atomic_write_bytes(path: Path, data: bytes) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + f".tmp.{os.getpid()}.{uuid.uuid4().hex}")
    try:
        tmp.write_bytes(data)
        tmp.replace(path)
        _chmod_private(path)
    finally:
        try:
            if tmp.exists():
                tmp.unlink()
        except Exception:
            pass


def encrypt_file_chunked(src: Path, dest: Path, key: bytes, aad: bytes = b"qroadscan-gemma4-model") -> None:
    aes = AESGCM(key)
    tmp = dest.with_suffix(dest.suffix + f".tmp.{uuid.uuid4().hex}")
    try:
        with src.open("rb") as source, tmp.open("wb") as out:
            out.write(FILE_ENCRYPTION_MAGIC)
            out.write(struct.pack(">I", FILE_ENCRYPTION_CHUNK_SIZE))
            index = 0
            while True:
                chunk = source.read(FILE_ENCRYPTION_CHUNK_SIZE)
                if not chunk:
                    break
                nonce = secrets.token_bytes(12)
                encrypted = aes.encrypt(nonce, chunk, aad + struct.pack(">Q", index))
                out.write(struct.pack(">I", len(encrypted)))
                out.write(nonce)
                out.write(encrypted)
                index += 1
        tmp.replace(dest)
        _chmod_private(dest)
    finally:
        if tmp.exists():
            tmp.unlink()


def decrypt_file_chunked(src: Path, dest: Path, key: bytes, aad: bytes = b"qroadscan-gemma4-model") -> None:
    aes = AESGCM(key)
    tmp = dest.with_suffix(dest.suffix + f".tmp.{uuid.uuid4().hex}")
    try:
        with src.open("rb") as handle, tmp.open("wb") as out:
            header = handle.read(len(FILE_ENCRYPTION_MAGIC))
            if header != FILE_ENCRYPTION_MAGIC:
                raise ValueError("Encrypted model header mismatch.")
            chunk_size_raw = handle.read(4)
            if len(chunk_size_raw) != 4:
                raise ValueError("Encrypted model chunk size is truncated.")
            chunk_size = struct.unpack(">I", chunk_size_raw)[0]
            if chunk_size <= 0 or chunk_size > 64 * 1024 * 1024:
                raise ValueError("Encrypted model chunk size is invalid.")
            index = 0
            while True:
                length_raw = handle.read(4)
                if not length_raw:
                    break
                if len(length_raw) != 4:
                    raise ValueError("Encrypted model chunk length is truncated.")
                encrypted_len = struct.unpack(">I", length_raw)[0]
                nonce = handle.read(12)
                payload = handle.read(encrypted_len)
                if len(nonce) != 12 or len(payload) != encrypted_len:
                    raise ValueError("Encrypted model payload is truncated.")
                out.write(aes.decrypt(nonce, payload, aad + struct.pack(">Q", index)))
                index += 1
        tmp.replace(dest)
        _chmod_private(dest)
    finally:
        if tmp.exists():
            tmp.unlink()


def append_model_ledger(event: str, data: Dict[str, Any]) -> None:
    rec = {
        "ts": time.time(),
        "event": event,
        "data": data,
    }
    line = json.dumps(rec, ensure_ascii=False, sort_keys=True) + "\n"
    with MODEL_LEDGER_PATH.open("a", encoding="utf-8") as f:
        f.write(line)
    _chmod_private(MODEL_LEDGER_PATH)


def download_model_httpx(url: str, dest: Path, *, expected_sha: Optional[str] = None) -> str:
    digest = hashlib.sha256()
    tmp = dest.with_suffix(dest.suffix + f".download.{uuid.uuid4().hex}")
    headers = model_download_headers()
    try:
        with httpx.stream("GET", url, headers=headers, follow_redirects=True, timeout=NETWORK_TIMEOUT) as response:
            response.raise_for_status()
            total = int(response.headers.get("Content-Length") or 0)
            done = 0
            last_print = 0.0
            with tmp.open("wb") as handle:
                for chunk in response.iter_bytes(chunk_size=1024 * 256):
                    if not chunk:
                        continue
                    handle.write(chunk)
                    digest.update(chunk)
                    done += len(chunk)
                    now = time.time()
                    if now - last_print > 3:
                        print(f"downloaded {human_size(done)} / {human_size(total) if total else 'unknown'}")
                        last_print = now
        sha = digest.hexdigest()
        if expected_sha and sha.lower() != expected_sha.lower():
            raise ValueError(f"Downloaded model SHA mismatch. Expected {expected_sha}, got {sha}.")
        tmp.replace(dest)
        _chmod_private(dest)
        return sha
    finally:
        if tmp.exists():
            tmp.unlink()


def path_size_bytes(path: Path) -> int:
    try:
        return path.stat().st_size if path.is_file() else 0
    except Exception:
        return 0


def disk_usage_report() -> Dict[str, Any]:
    usage = shutil.disk_usage(str(ROOT_DIR if ROOT_DIR.exists() else Path("/content")))
    return {
        "total": human_size(usage.total),
        "used": human_size(usage.used),
        "free": human_size(usage.free),
        "plain_model": human_size(path_size_bytes(PLAIN_MODEL_PATH)),
        "encrypted_model": human_size(path_size_bytes(ENCRYPTED_MODEL_PATH)),
    }


def cleanup_stale_model_artifacts() -> None:
    if not bool(globals().get("CLEANUP_STALE_MODEL_ARTIFACTS", True)):
        return
    removed = []
    patterns = [
        MODELS_DIR / f"{MODEL_FILE}.download.*",
        MODELS_DIR / f"{MODEL_FILE}.tmp.*",
        TEMP_DIR / "*.litertlm",
        TEMP_DIR / "*.tmp.*",
        CACHE_DIR / "*",
    ]
    for pattern in patterns:
        for p in pattern.parent.glob(pattern.name):
            try:
                if p.is_dir():
                    shutil.rmtree(p, ignore_errors=True)
                elif p.is_file():
                    p.unlink()
                removed.append(str(p))
            except Exception:
                pass
    if removed:
        append_model_ledger("cleanup_stale_artifacts", {"removed_count": len(removed), "sample": removed[:12]})
        print(f"Cleaned {len(removed)} stale model/cache artifact(s).")


def configured_model_engine() -> str:
    engine = str(globals().get("MODEL_ENGINE", "transformers-cuda") or "transformers-cuda").strip().lower()
    aliases = {"cuda": "transformers-cuda", "transformers": "transformers-cuda", "litert": "litert-gpu"}
    engine = aliases.get(engine, engine)
    if engine not in {"transformers-cuda", "auto", "litert-gpu", "litert-cpu"}:
        raise ValueError("MODEL_ENGINE must be transformers-cuda, auto, litert-gpu, or litert-cpu.")
    return engine


def purge_litert_artifacts_for_transformers() -> None:
    if not bool(globals().get("PURGE_LITERT_ARTIFACTS_WHEN_TRANSFORMERS", True)):
        return
    removed = []
    candidates = [
        PLAIN_MODEL_PATH,
        ENCRYPTED_MODEL_PATH,
        PLAIN_MODEL_PATH.with_suffix(PLAIN_MODEL_PATH.suffix + ".verified.json"),
    ]
    for p in candidates:
        try:
            if p.exists() and p.is_file():
                removed.append({"path": str(p), "bytes": path_size_bytes(p)})
                p.unlink()
        except Exception as exc:
            removed.append({"path": str(p), "error": repr(exc)})
    if removed:
        append_model_ledger("purged_litert_artifacts_for_transformers", {"removed": removed})
        print("Purged LiteRT artifacts because MODEL_ENGINE='transformers-cuda':", removed)


def model_storage_mode() -> str:
    mode = str(globals().get("MODEL_STORAGE_MODE", "plain") or "plain").strip().lower()
    if mode not in {"plain", "encrypted"}:
        raise ValueError("MODEL_STORAGE_MODE must be 'plain' or 'encrypted'.")
    return mode


def verify_plain_model(path: Path, *, force_hash: bool = False) -> Tuple[bool, str]:
    if not path.exists():
        return False, "missing"
    if path_size_bytes(path) <= 0:
        return False, "empty"
    marker = path.with_suffix(path.suffix + ".verified.json")
    if marker.exists() and not force_hash:
        try:
            data = json.loads(marker.read_text(encoding="utf-8"))
            if data.get("sha256") == EXPECTED_HASH and data.get("bytes") == path_size_bytes(path):
                return True, "verified-marker"
        except Exception:
            pass
    sha = sha256_file(path)
    ok = sha.lower() == EXPECTED_HASH.lower()
    if ok:
        marker.write_text(json.dumps({"sha256": sha, "bytes": path_size_bytes(path), "verified_ts": time.time()}, indent=2), encoding="utf-8")
        _chmod_private(marker)
        return True, "sha256"
    return False, f"sha-mismatch:{sha}"


def ensure_plain_model_from_download() -> None:
    print("Downloading verified plaintext LiteRT-LM model directly to the runtime path...")
    sha = download_model_httpx(MODEL_REPO + MODEL_FILE, PLAIN_MODEL_PATH, expected_sha=EXPECTED_HASH)
    marker = PLAIN_MODEL_PATH.with_suffix(PLAIN_MODEL_PATH.suffix + ".verified.json")
    marker.write_text(json.dumps({"sha256": sha, "bytes": path_size_bytes(PLAIN_MODEL_PATH), "verified_ts": time.time()}, indent=2), encoding="utf-8")
    _chmod_private(marker)
    append_model_ledger("download_verified_plain", {"sha256": sha, "path": str(PLAIN_MODEL_PATH), "bytes": path_size_bytes(PLAIN_MODEL_PATH)})


def ensure_plain_model() -> Optional[bytes]:
    force_hash = bool(globals().get("MODEL_VERIFY_EACH_RUN", False))
    ok, reason = verify_plain_model(PLAIN_MODEL_PATH, force_hash=force_hash)
    if ok:
        print(f"Plain LiteRT model ready ({reason}): {PLAIN_MODEL_PATH}")
    else:
        if ENCRYPTED_MODEL_PATH.exists() and bool(globals().get("ALLOW_DECRYPT_ENCRYPTED_TO_PLAIN_ON_MODE_SWITCH", False)):
            print("Plain model missing; attempting one-time decrypt from existing encrypted artifact...")
            try:
                key = vault_key_from_passphrase(model_vault_passphrase())
                decrypt_file_chunked(ENCRYPTED_MODEL_PATH, PLAIN_MODEL_PATH, key)
                ok, reason = verify_plain_model(PLAIN_MODEL_PATH, force_hash=True)
                if not ok:
                    raise ValueError(f"Decrypted plaintext model failed verification: {reason}")
                append_model_ledger("decrypted_to_plain_runtime", {"path": str(PLAIN_MODEL_PATH), "reason": reason})
            except Exception as exc:
                print("Encrypted artifact could not be reused; downloading a clean verified plaintext model instead:", repr(exc))
                try:
                    if PLAIN_MODEL_PATH.exists():
                        PLAIN_MODEL_PATH.unlink()
                except Exception:
                    pass
                ensure_plain_model_from_download()
        else:
            ensure_plain_model_from_download()

    if ENCRYPTED_MODEL_PATH.exists() and bool(globals().get("PURGE_ENCRYPTED_MODEL_WHEN_PLAIN_MODE", True)):
        try:
            ENCRYPTED_MODEL_PATH.unlink()
            append_model_ledger("purged_encrypted_in_plain_mode", {"path": str(ENCRYPTED_MODEL_PATH)})
            print("Removed encrypted duplicate because MODEL_STORAGE_MODE='plain'.")
        except Exception as exc:
            print("Could not remove encrypted duplicate:", repr(exc))
    return None


def ensure_encrypted_model() -> bytes:
    passphrase = model_vault_passphrase()
    key = vault_key_from_passphrase(passphrase)
    force_hash = bool(globals().get("MODEL_VERIFY_EACH_RUN", False))

    if ENCRYPTED_MODEL_PATH.exists():
        if force_hash:
            print("Encrypted model exists. Verifying by temporary decrypt + SHA-256...")
            temp_plain = TEMP_DIR / f"verify.{os.getpid()}.{uuid.uuid4().hex}.litertlm"
            try:
                decrypt_file_chunked(ENCRYPTED_MODEL_PATH, temp_plain, key)
                sha = sha256_file(temp_plain)
                ok = sha.lower() == EXPECTED_HASH.lower()
                append_model_ledger("verify_encrypted", {"sha256": sha, "ok": ok, "path": str(ENCRYPTED_MODEL_PATH)})
                if not ok:
                    raise ValueError(f"Encrypted model SHA mismatch. Expected {EXPECTED_HASH}, got {sha}.")
            finally:
                if temp_plain.exists():
                    temp_plain.unlink()
        print("Encrypted LiteRT model ready:", ENCRYPTED_MODEL_PATH)
        return key

    ok, reason = verify_plain_model(PLAIN_MODEL_PATH, force_hash=force_hash)
    if not ok:
        ensure_plain_model_from_download()
    print("Encrypting verified plaintext model with AES-GCM...")
    encrypt_file_chunked(PLAIN_MODEL_PATH, ENCRYPTED_MODEL_PATH, key)
    append_model_ledger("encrypted_model", {"encrypted_path": str(ENCRYPTED_MODEL_PATH), "bytes": path_size_bytes(ENCRYPTED_MODEL_PATH)})
    if bool(globals().get("DELETE_PLAINTEXT_AFTER_ENCRYPT", True)):
        try:
            PLAIN_MODEL_PATH.unlink()
            marker = PLAIN_MODEL_PATH.with_suffix(PLAIN_MODEL_PATH.suffix + ".verified.json")
            if marker.exists():
                marker.unlink()
            append_model_ledger("plain_removed_after_encrypt", {"path": str(PLAIN_MODEL_PATH)})
        except FileNotFoundError:
            pass
    return key


MODEL_RUNTIME_INFO: Dict[str, Any] = {}


def prepare_or_verify_model() -> Optional[bytes]:
    cleanup_stale_model_artifacts()
    print("Disk before model setup:", json.dumps(disk_usage_report(), ensure_ascii=False))
    engine = configured_model_engine()
    if engine == "transformers-cuda":
        purge_litert_artifacts_for_transformers()
        MODEL_RUNTIME_INFO.update({
            "engine": engine,
            "storage_mode": "hf-transformers-cache",
            "transformers_model_id": str(globals().get("TRANSFORMERS_MODEL_ID", "google/gemma-4-E2B-it")),
            "transformers_cache_dir": str(globals().get("TRANSFORMERS_CACHE_DIR", "")),
            "litert_model_skipped": True,
            "plain_model_path": str(PLAIN_MODEL_PATH),
            "encrypted_model_path": str(ENCRYPTED_MODEL_PATH),
            "plain_exists": PLAIN_MODEL_PATH.exists(),
            "encrypted_exists": ENCRYPTED_MODEL_PATH.exists(),
            "disk": disk_usage_report(),
        })
        append_model_ledger("model_runtime_ready_transformers", MODEL_RUNTIME_INFO)
        print("Skipping LiteRT .litertlm download because MODEL_ENGINE='transformers-cuda'.")
        print("Model runtime info:", json.dumps(MODEL_RUNTIME_INFO, ensure_ascii=False, indent=2))
        return None

    mode = model_storage_mode()
    key = ensure_plain_model() if mode == "plain" else ensure_encrypted_model()
    runtime_path = PLAIN_MODEL_PATH if mode == "plain" else ENCRYPTED_MODEL_PATH
    MODEL_RUNTIME_INFO.update({
        "engine": engine,
        "storage_mode": mode,
        "runtime_path": str(runtime_path),
        "plain_model_path": str(PLAIN_MODEL_PATH),
        "encrypted_model_path": str(ENCRYPTED_MODEL_PATH),
        "plain_exists": PLAIN_MODEL_PATH.exists(),
        "encrypted_exists": ENCRYPTED_MODEL_PATH.exists(),
        "disk": disk_usage_report(),
    })
    append_model_ledger("model_runtime_ready", MODEL_RUNTIME_INFO)
    print("Model runtime info:", json.dumps(MODEL_RUNTIME_INFO, ensure_ascii=False, indent=2))
    return key


VAULT_KEY = prepare_or_verify_model()
if configured_model_engine() == "transformers-cuda":
    print("CUDA Transformers runtime will load the model from Hugging Face cache onto the T4.")
elif model_storage_mode() == "plain":
    print("Plain verified model will be loaded directly by LiteRT-LM. No vault key needed for runtime.")
else:
    print("Model vault key is in memory for this runtime only.")


In [ ]:
# Cell 3 — model runtime wrapper: CUDA Transformers on Colab, LiteRT optional

from __future__ import annotations

import contextlib
import os
import shutil
import subprocess
import sys
import threading
import time
import uuid
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

litert_lm = None
LITERT_IMPORT_ERROR = None
LITERT_RUNTIME_AUDIT: Dict[str, Any] = {}
TRANSFORMERS_RUNTIME_AUDIT: Dict[str, Any] = {}
ACTIVE_MODEL_ENGINE: Optional[str] = None
CUDA_TRANSFORMERS_RUNTIME = None
CUDA_TRANSFORMERS_RUNTIME_LOCK = threading.RLock()


def require_litert_lm() -> None:
    global litert_lm, LITERT_IMPORT_ERROR
    if litert_lm is None and LITERT_IMPORT_ERROR is None:
        try:
            import litert_lm as litert_lm_module
        except Exception as exc:
            LITERT_IMPORT_ERROR = exc
        else:
            litert_lm = litert_lm_module
    if litert_lm is None:
        detail = f" Import error: {LITERT_IMPORT_ERROR}" if LITERT_IMPORT_ERROR else ""
        raise RuntimeError("LiteRT-LM is not installed in this Colab runtime." + detail)


def nvidia_smi_snapshot() -> Dict[str, Any]:
    try:
        proc = subprocess.run(
            [
                "nvidia-smi",
                "--query-gpu=name,memory.total,memory.used,memory.free,utilization.gpu",
                "--format=csv,noheader,nounits",
            ],
            capture_output=True,
            text=True,
            timeout=4,
            check=False,
        )
        if proc.returncode != 0 or not proc.stdout.strip():
            return {"available": False, "reason": (proc.stderr or "nvidia-smi returned no GPU").strip()[:240]}
        name, total, used, free, util = [x.strip() for x in proc.stdout.strip().splitlines()[0].split(",")[:5]]
        return {
            "available": True,
            "name": name,
            "total_gb": round(float(total) / 1024.0, 3),
            "used_gb": round(float(used) / 1024.0, 3),
            "free_gb": round(float(free) / 1024.0, 3),
            "gpu_util_percent": float(util),
        }
    except Exception as exc:
        return {"available": False, "reason": repr(exc)[:240]}


def require_cuda_runtime() -> Dict[str, Any]:
    snap = nvidia_smi_snapshot()
    TRANSFORMERS_RUNTIME_AUDIT["gpu_preflight"] = snap
    if bool(globals().get("REQUIRE_GPU_RUNTIME", True)) and not snap.get("available"):
        raise RuntimeError(
            "CUDA Transformers runtime requested, but nvidia-smi is not available. "
            "In Colab: Runtime > Change runtime type > T4 GPU, then reconnect. "
            f"Detail: {snap}"
        )
    return snap


def _litert_backend_attr(*names: str) -> Any:
    require_litert_lm()
    for name in names:
        try:
            return getattr(litert_lm.Backend, name)
        except Exception:
            continue
    return None


def available_litert_backends() -> List[str]:
    try:
        require_litert_lm()
        return sorted(name for name in dir(litert_lm.Backend) if name.isupper() and not name.startswith("_"))
    except Exception:
        return []


def _litert_cpu_backend() -> Any:
    backend = _litert_backend_attr("CPU")
    if backend is None:
        raise RuntimeError("This LiteRT-LM build does not expose a CPU backend.")
    return backend


def _litert_gpu_backend() -> Any:
    backend = _litert_backend_attr("GPU", "GPU_FLOAT16", "GPU_FLOAT32")
    if backend is None:
        raise RuntimeError(
            "This LiteRT-LM build does not expose a GPU backend. "
            f"Available Backend attributes: {available_litert_backends()}"
        )
    return backend


def backend_value_for_name(name: str) -> Tuple[str, Any]:
    requested = (name or "GPU").strip().lower()
    allow_cpu = bool(globals().get("ALLOW_CPU_FALLBACK", False))
    if requested == "cpu":
        return "CPU", _litert_cpu_backend()
    if requested == "gpu":
        return "GPU", _litert_gpu_backend()
    if requested == "auto":
        try:
            return "GPU", _litert_gpu_backend()
        except Exception as gpu_exc:
            if allow_cpu:
                return "CPU", _litert_cpu_backend()
            raise RuntimeError("Auto backend could not initialize GPU and CPU fallback is disabled.") from gpu_exc
    raise ValueError("INFERENCE_BACKEND must be GPU, Auto, or CPU.")


def validate_gpu_request() -> None:
    requested = str(globals().get("INFERENCE_BACKEND", "GPU") or "GPU").lower()
    if requested not in {"gpu", "auto"}:
        return
    if not bool(globals().get("REQUIRE_GPU_RUNTIME", True)):
        return
    snap = nvidia_smi_snapshot()
    LITERT_RUNTIME_AUDIT["preflight_gpu"] = snap
    if not snap.get("available"):
        raise RuntimeError(
            "GPU runtime was requested but nvidia-smi is not available. "
            "In Colab, switch Runtime > Change runtime type > T4 GPU, then reconnect. "
            f"Detail: {snap}"
        )


def _safe_worker_token(worker_id: Optional[str]) -> str:
    token = "".join(ch for ch in str(worker_id or "solo") if ch.isalnum() or ch in "._-")[:36]
    return token or "solo"


@contextlib.contextmanager
def temporary_litert_cache(worker_id: Optional[str] = None):
    if not bool(globals().get("ENABLE_LITERT_CACHE", False)):
        yield None
        return
    worker = _safe_worker_token(worker_id)
    cache_path = CACHE_DIR / f"{worker}_{os.getpid()}_{time.time_ns()}_{uuid.uuid4().hex}"
    cache_path.mkdir(parents=True, exist_ok=False)
    try:
        yield cache_path
    finally:
        shutil.rmtree(cache_path, ignore_errors=True)


def requested_model_engine() -> str:
    if "configured_model_engine" in globals():
        return configured_model_engine()
    engine = str(globals().get("MODEL_ENGINE", "transformers-cuda") or "transformers-cuda").strip().lower()
    return {"cuda": "transformers-cuda", "transformers": "transformers-cuda", "litert": "litert-gpu"}.get(engine, engine)


@contextlib.contextmanager
def unlocked_model_path(key: Optional[bytes], *, label: str = "gemma_model"):
    engine = ACTIVE_MODEL_ENGINE or requested_model_engine()
    if engine == "transformers-cuda":
        yield None
        return
    mode = model_storage_mode() if "model_storage_mode" in globals() else "plain"
    if mode == "plain":
        ok, reason = verify_plain_model(PLAIN_MODEL_PATH, force_hash=bool(globals().get("MODEL_VERIFY_EACH_RUN", False)))
        if not ok:
            raise RuntimeError(f"Plain model is not ready for LiteRT load: {reason}")
        yield PLAIN_MODEL_PATH
        return
    if key is None:
        raise RuntimeError("Encrypted model mode requires VAULT_KEY in memory.")
    safe_label = _safe_worker_token(label)
    temp_model = TEMP_DIR / f"{safe_label}.{os.getpid()}.{uuid.uuid4().hex}.litertlm"
    try:
        decrypt_file_chunked(ENCRYPTED_MODEL_PATH, temp_model, key)
        yield temp_model
    finally:
        try:
            if temp_model.exists():
                temp_model.unlink()
        except Exception:
            pass


def create_default_messages(system_text: Optional[str] = None) -> List[dict]:
    if not system_text:
        return []
    return [{"role": "system", "content": [{"type": "text", "text": sanitize_text(system_text, max_chars=18000)}]}]


def create_user_message(user_text: str) -> Any:
    return sanitize_text(user_text, max_chars=MAX_PROMPT_CHARS)


def response_to_text(response: Any) -> str:
    if not isinstance(response, dict):
        return sanitize_text(response, max_chars=MAX_ARTICLE_CHARS)
    parts = response.get("content", [])
    texts: List[str] = []
    for item in parts:
        if isinstance(item, dict) and item.get("type") == "text":
            texts.append(str(item.get("text", "")))
    return sanitize_text("".join(texts), max_chars=MAX_ARTICLE_CHARS)


def load_litert_engine(model_path: Path, cache_dir: Optional[Path] = None, *, inference_backend: str = "GPU"):
    require_litert_lm()
    validate_gpu_request()
    try:
        litert_lm.set_min_log_severity(litert_lm.LogSeverity.ERROR)
    except Exception:
        pass
    backend_label, backend_value = backend_value_for_name(inference_backend)
    engine_kwargs = {"backend": backend_value}
    if cache_dir is not None:
        engine_kwargs["cache_dir"] = str(cache_dir)
    before = nvidia_smi_snapshot()
    try:
        engine = litert_lm.Engine(str(model_path), **engine_kwargs)
    except Exception as exc:
        if backend_label == "GPU":
            raise RuntimeError(
                "LiteRT-LM failed to construct a GPU engine. CPU fallback is disabled so this does not silently run on CPU. "
                "On this Colab/T4, use MODEL_ENGINE='transformers-cuda' to run Gemma on CUDA. "
                f"Available backends: {available_litert_backends()}. Original error: {exc!r}"
            ) from exc
        raise
    LITERT_RUNTIME_AUDIT.update({
        "selected_backend": backend_label,
        "backend_value": repr(backend_value),
        "model_path": str(model_path),
        "cache_dir": str(cache_dir) if cache_dir else None,
        "gpu_before_engine_construct": before,
    })
    return engine


def probe_litert_gpu_once() -> bool:
    try:
        with unlocked_model_path(VAULT_KEY, label="litert_gpu_probe") as model_path:
            if model_path is None:
                return False
            with temporary_litert_cache("litert_probe") as cache_dir:
                engine = load_litert_engine(Path(model_path), cache_dir=cache_dir, inference_backend="GPU")
                try:
                    engine.__enter__()
                finally:
                    engine.__exit__(None, None, None)
        LITERT_RUNTIME_AUDIT["gpu_probe_ok"] = True
        return True
    except Exception as exc:
        LITERT_RUNTIME_AUDIT["gpu_probe_ok"] = False
        LITERT_RUNTIME_AUDIT["gpu_probe_error"] = repr(exc)
        print("LiteRT GPU probe failed; routing to CUDA Transformers if configured:", repr(exc))
        return False


def resolve_active_model_engine() -> str:
    global ACTIVE_MODEL_ENGINE
    if ACTIVE_MODEL_ENGINE:
        return ACTIVE_MODEL_ENGINE
    requested = requested_model_engine()
    if requested == "transformers-cuda":
        require_cuda_runtime()
        ACTIVE_MODEL_ENGINE = "transformers-cuda"
    elif requested == "litert-cpu":
        ACTIVE_MODEL_ENGINE = "litert-cpu"
    elif requested == "litert-gpu":
        ACTIVE_MODEL_ENGINE = "litert-gpu"
    elif requested == "auto":
        ACTIVE_MODEL_ENGINE = "litert-gpu" if probe_litert_gpu_once() else str(globals().get("LITERT_GPU_FALLBACK_ENGINE", "transformers-cuda"))
        if ACTIVE_MODEL_ENGINE == "transformers-cuda":
            require_cuda_runtime()
    else:
        raise ValueError(f"Unknown MODEL_ENGINE: {requested}")
    LITERT_RUNTIME_AUDIT["active_model_engine"] = ACTIVE_MODEL_ENGINE
    TRANSFORMERS_RUNTIME_AUDIT["active_model_engine"] = ACTIVE_MODEL_ENGINE
    print("Active model engine:", ACTIVE_MODEL_ENGINE)
    return ACTIVE_MODEL_ENGINE


class CudaTransformersRuntime:
    def __init__(self):
        self.model = None
        self.processor = None
        self.tokenizer = None
        self.lock = threading.RLock()
        self.generate_sem = threading.Semaphore(max(1, int(globals().get("MAX_TRANSFORMERS_CONCURRENT_GENERATIONS", 1) or 1)))
        self.loaded = False
        self.model_id = str(globals().get("TRANSFORMERS_MODEL_ID", "google/gemma-4-E2B-it"))

    def _torch_dtype(self, torch_module: Any) -> Any:
        value = str(globals().get("TRANSFORMERS_DTYPE", "float16") or "float16").lower()
        if value in {"float16", "fp16", "half"}:
            return torch_module.float16
        if value in {"bfloat16", "bf16"}:
            return torch_module.bfloat16
        if value in {"float32", "fp32"}:
            return torch_module.float32
        return torch_module.float16

    def load_once(self) -> None:
        if self.loaded:
            return
        with self.lock:
            if self.loaded:
                return
            require_cuda_runtime()
            import torch
            from transformers import AutoModelForCausalLM, AutoProcessor, AutoTokenizer
            try:
                from transformers import BitsAndBytesConfig
            except Exception:
                BitsAndBytesConfig = None

            if not torch.cuda.is_available():
                raise RuntimeError("torch.cuda.is_available() is False even though CUDA Transformers was requested.")

            cache_dir = str(globals().get("TRANSFORMERS_CACHE_DIR", "") or "").strip() or None
            if cache_dir:
                Path(cache_dir).mkdir(parents=True, exist_ok=True)
                os.environ.setdefault("HF_HOME", cache_dir)
                os.environ.setdefault("TRANSFORMERS_CACHE", cache_dir)
                os.environ.setdefault("HF_HUB_CACHE", str(Path(cache_dir) / "hub"))

            token, token_source = huggingface_token() if "huggingface_token" in globals() else ("", "not_configured")
            if token:
                os.environ.setdefault("HF_TOKEN", token)

            dtype = self._torch_dtype(torch)
            load_in_4bit = bool(globals().get("TRANSFORMERS_LOAD_IN_4BIT", False))
            trust_remote_code = bool(globals().get("TRANSFORMERS_TRUST_REMOTE_CODE", True))
            device_map_setting = str(globals().get("TRANSFORMERS_DEVICE_MAP", "cuda") or "cuda").lower()
            device_map: Any = {"": 0} if device_map_setting == "cuda" else device_map_setting

            common_kwargs = {"cache_dir": cache_dir, "token": token or None, "trust_remote_code": trust_remote_code}
            common_kwargs = {k: v for k, v in common_kwargs.items() if v is not None}

            before = nvidia_smi_snapshot()
            print(f"Loading {self.model_id} with Transformers onto CUDA. 4bit={load_in_4bit}, device_map={device_map_setting}")
            self.processor = AutoProcessor.from_pretrained(self.model_id, **common_kwargs)
            try:
                self.tokenizer = getattr(self.processor, "tokenizer", None) or AutoTokenizer.from_pretrained(self.model_id, **common_kwargs)
            except Exception:
                self.tokenizer = self.processor

            model_kwargs = dict(common_kwargs)
            model_kwargs.update({"torch_dtype": dtype, "device_map": device_map, "low_cpu_mem_usage": True})
            if load_in_4bit:
                if BitsAndBytesConfig is None:
                    raise RuntimeError("TRANSFORMERS_LOAD_IN_4BIT=True but BitsAndBytesConfig is unavailable. Install bitsandbytes or set 4bit=False.")
                model_kwargs["quantization_config"] = BitsAndBytesConfig(
                    load_in_4bit=True,
                    bnb_4bit_quant_type="nf4",
                    bnb_4bit_use_double_quant=True,
                    bnb_4bit_compute_dtype=dtype,
                )
                model_kwargs.pop("torch_dtype", None)

            self.model = AutoModelForCausalLM.from_pretrained(self.model_id, **model_kwargs)
            self.model.eval()
            torch.cuda.synchronize()
            after = nvidia_smi_snapshot()
            TRANSFORMERS_RUNTIME_AUDIT.update({
                "model_id": self.model_id,
                "cache_dir": cache_dir,
                "token_source": token_source,
                "load_in_4bit": load_in_4bit,
                "device_map": device_map_setting,
                "dtype": str(dtype),
                "gpu_before_load": before,
                "gpu_after_load": after,
                "torch_cuda_device": torch.cuda.get_device_name(0),
            })
            self.loaded = True
            print("CUDA Transformers model loaded. GPU:", after)

    def _format_prompt(self, user_text: str, system_text: Optional[str]) -> str:
        system_text = sanitize_text(system_text or "", max_chars=18000)
        user_text = sanitize_text(user_text, max_chars=MAX_PROMPT_CHARS)
        messages = []
        if system_text:
            messages.append({"role": "system", "content": system_text})
        messages.append({"role": "user", "content": user_text})
        templater = self.processor if hasattr(self.processor, "apply_chat_template") else self.tokenizer
        if hasattr(templater, "apply_chat_template"):
            return templater.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        prefix = f"System:\n{system_text}\n\n" if system_text else ""
        return prefix + f"User:\n{user_text}\n\nAssistant:\n"

    def chat(self, user_text: str, *, system_text: Optional[str] = None) -> str:
        self.load_once()
        import torch
        prompt = self._format_prompt(user_text, system_text)
        tokenizer = self.tokenizer
        with self.generate_sem:
            inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=8192)
            target_device = next(self.model.parameters()).device
            inputs = {k: v.to(target_device) for k, v in inputs.items()}
            input_len = int(inputs["input_ids"].shape[-1])
            gen_kwargs = {
                "max_new_tokens": int(globals().get("TRANSFORMERS_MAX_NEW_TOKENS", 1800) or 1800),
                "do_sample": True,
                "temperature": float(globals().get("TRANSFORMERS_TEMPERATURE", 0.72) or 0.72),
                "top_p": float(globals().get("TRANSFORMERS_TOP_P", 0.92) or 0.92),
            }
            if getattr(tokenizer, "pad_token_id", None) is not None:
                gen_kwargs["pad_token_id"] = tokenizer.pad_token_id
            elif getattr(tokenizer, "eos_token_id", None) is not None:
                gen_kwargs["pad_token_id"] = tokenizer.eos_token_id
            before = nvidia_smi_snapshot()
            with torch.inference_mode():
                output = self.model.generate(**inputs, **gen_kwargs)
            torch.cuda.synchronize()
            after = nvidia_smi_snapshot()
            TRANSFORMERS_RUNTIME_AUDIT["last_generate_gpu_before"] = before
            TRANSFORMERS_RUNTIME_AUDIT["last_generate_gpu_after"] = after
            new_tokens = output[0][input_len:]
            text = tokenizer.decode(new_tokens, skip_special_tokens=True)
            return sanitize_text(text, max_chars=MAX_ARTICLE_CHARS)


def get_cuda_transformers_runtime() -> CudaTransformersRuntime:
    global CUDA_TRANSFORMERS_RUNTIME
    with CUDA_TRANSFORMERS_RUNTIME_LOCK:
        if CUDA_TRANSFORMERS_RUNTIME is None:
            CUDA_TRANSFORMERS_RUNTIME = CudaTransformersRuntime()
        return CUDA_TRANSFORMERS_RUNTIME


class Gemma4E2BSession:
    """Unified session interface backed by CUDA Transformers or LiteRT."""

    def __init__(
        self,
        key: Optional[bytes] = None,
        *,
        inference_backend: str = "GPU",
        worker_id: str = "solo",
        model_path_override: Optional[Path] = None,
    ):
        self.key = key
        self.inference_backend = inference_backend
        self.worker_id = worker_id
        self.model_path_override = Path(model_path_override) if model_path_override else None
        self._model_cm = None
        self._cache_cm = None
        self.model_path = None
        self.cache_dir = None
        self.engine = None
        self.transformers_runtime = None
        self.active_engine = None
        self.call_count = 0
        self.total_seconds = 0.0
        self.gpu_before_enter: Dict[str, Any] = {}
        self.gpu_after_enter: Dict[str, Any] = {}

    def __enter__(self):
        self.active_engine = resolve_active_model_engine()
        if self.active_engine == "transformers-cuda":
            self.gpu_before_enter = nvidia_smi_snapshot()
            self.transformers_runtime = get_cuda_transformers_runtime()
            self.transformers_runtime.load_once()
            self.gpu_after_enter = nvidia_smi_snapshot()
            print(f"{self.worker_id} CUDA Transformers session ready. GPU={self.gpu_after_enter}")
            return self

        try:
            if self.model_path_override is not None:
                self.model_path = self.model_path_override
            else:
                self._model_cm = unlocked_model_path(self.key, label=f"{self.worker_id}_model")
                self.model_path = self._model_cm.__enter__()
            self._cache_cm = temporary_litert_cache(self.worker_id)
            self.cache_dir = self._cache_cm.__enter__()
            self.gpu_before_enter = nvidia_smi_snapshot()
            backend = "CPU" if self.active_engine == "litert-cpu" else self.inference_backend
            self.engine = load_litert_engine(self.model_path, cache_dir=self.cache_dir, inference_backend=backend)
            try:
                self.engine.__enter__()
            except Exception as exc:
                if str(backend or "GPU").lower() in {"gpu", "auto"}:
                    raise RuntimeError(
                        "LiteRT-LM constructed an engine but failed while entering/loading it on GPU. "
                        "Use MODEL_ENGINE='transformers-cuda' on Colab/T4. "
                        f"GPU before enter: {self.gpu_before_enter}. Original error: {exc!r}"
                    ) from exc
                raise
            self.gpu_after_enter = nvidia_smi_snapshot()
            print(
                f"{self.worker_id} LiteRT engine ready: backend={LITERT_RUNTIME_AUDIT.get('selected_backend')} "
                f"model={self.model_path} cache={self.cache_dir or 'disabled'} gpu={self.gpu_after_enter}"
            )
            return self
        except Exception:
            try:
                if self.engine is not None:
                    self.engine.__exit__(*sys.exc_info())
            except Exception:
                pass
            try:
                if self._cache_cm is not None:
                    self._cache_cm.__exit__(*sys.exc_info())
            except Exception:
                pass
            try:
                if self._model_cm is not None:
                    self._model_cm.__exit__(*sys.exc_info())
            except Exception:
                pass
            raise

    def __exit__(self, exc_type, exc, tb):
        if self.active_engine == "transformers-cuda":
            return False
        try:
            if self.engine is not None:
                self.engine.__exit__(exc_type, exc, tb)
        finally:
            if self._cache_cm is not None:
                self._cache_cm.__exit__(exc_type, exc, tb)
            if self._model_cm is not None:
                self._model_cm.__exit__(exc_type, exc, tb)
        return False

    def chat(self, user_text: str, *, system_text: Optional[str] = None, retries: int = 1) -> str:
        last_exc = None
        for attempt in range(max(1, retries + 1)):
            started = time.time()
            try:
                if self.active_engine == "transformers-cuda":
                    text = self.transformers_runtime.chat(user_text, system_text=system_text)
                else:
                    if self.engine is None:
                        raise RuntimeError("Gemma4E2BSession is not open.")
                    messages = create_default_messages(system_text)
                    with self.engine.create_conversation(messages=messages) as conversation:
                        text = response_to_text(conversation.send_message(create_user_message(user_text)))
                self.call_count += 1
                self.total_seconds += time.time() - started
                return text
            except Exception as exc:
                last_exc = exc
                if attempt >= retries:
                    break
                time.sleep(0.6 + attempt * 0.4)
        raise RuntimeError(f"{self.worker_id} model chat failed after retry: {last_exc}")


try:
    require_litert_lm()
    LITERT_RUNTIME_AUDIT["available_backends"] = available_litert_backends()
except Exception as exc:
    LITERT_RUNTIME_AUDIT["available_backends_error"] = repr(exc)
LITERT_RUNTIME_AUDIT["gpu_preflight"] = nvidia_smi_snapshot()
TRANSFORMERS_RUNTIME_AUDIT["gpu_preflight"] = nvidia_smi_snapshot()
print("Unified model runtime ready.")
print("Requested MODEL_ENGINE:", requested_model_engine())
print("Available LiteRT backends:", LITERT_RUNTIME_AUDIT.get("available_backends"))
print("GPU preflight:", TRANSFORMERS_RUNTIME_AUDIT["gpu_preflight"])


In [ ]:
# Cell 4 — Quantum RGB simulation + advanced local RAG citation surfaces

from __future__ import annotations

import colorsys
import csv
import hashlib
import json
import math
import os
import random
import re
import secrets
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import psutil

try:
    import pennylane as qml
    from pennylane import numpy as pnp
except Exception:
    qml = None
    pnp = None

WORD_RE = re.compile(r"[A-Za-z0-9][A-Za-z0-9_\-]{1,}")
SENTENCE_RE = re.compile(r"(?<=[.!?])\s+")


def stable_hash(text: str, n: int = 12) -> str:
    return hashlib.blake2s(str(text).encode("utf-8"), digest_size=16).hexdigest()[:n]


def slugify(text: str, max_len: int = 90) -> str:
    text = re.sub(r"[^A-Za-z0-9]+", "-", str(text).lower()).strip("-")
    return (text[:max_len].strip("-") or "article")


def tokens(text: str) -> List[str]:
    return [m.group(0).lower() for m in WORD_RE.finditer(str(text))]


def split_chunks(text: str, *, max_words: int = 170, overlap: int = 35) -> List[str]:
    words = str(text).split()
    if not words:
        return []
    chunks = []
    step = max(1, max_words - overlap)
    for start in range(0, len(words), step):
        chunk = " ".join(words[start : start + max_words]).strip()
        if chunk:
            chunks.append(chunk)
        if start + max_words >= len(words):
            break
    return chunks


def read_text_file(path: Path) -> str:
    try:
        if path.suffix.lower() == ".json":
            return json.dumps(json.loads(path.read_text(encoding="utf-8")), ensure_ascii=False, indent=2)
        if path.suffix.lower() == ".csv":
            rows = []
            with path.open("r", encoding="utf-8", errors="ignore", newline="") as f:
                reader = csv.reader(f)
                for i, row in enumerate(reader):
                    rows.append(" | ".join(row))
                    if i > 2000:
                        break
            return "\n".join(rows)
        return path.read_text(encoding="utf-8", errors="ignore")
    except Exception as exc:
        return f"[unreadable file {path.name}: {exc}]"


@dataclass
class SurfaceChunk:
    surface_id: str
    chunk_id: str
    title: str
    url: str
    text: str
    fingerprint: str
    vector: np.ndarray

    @property
    def cite(self) -> str:
        return f"⟦SFC:{self.surface_id}#{self.chunk_id}@{self.fingerprint}⟧"


class HashedVectorizer:
    def __init__(self, dims: int = 384):
        self.dims = int(dims)

    def transform_one(self, text: str) -> np.ndarray:
        vec = np.zeros(self.dims, dtype=np.float32)
        toks = tokens(text)
        if not toks:
            return vec
        for tok in toks:
            h = int(hashlib.blake2b(tok.encode(), digest_size=8).hexdigest(), 16)
            idx = h % self.dims
            sign = 1.0 if ((h >> 11) & 1) else -1.0
            vec[idx] += sign * (1.0 + min(len(tok), 12) / 16.0)
        norm = float(np.linalg.norm(vec))
        return vec / norm if norm else vec


class CitationSurfaceIndex:
    def __init__(self, vectorizer: Optional[HashedVectorizer] = None):
        self.vectorizer = vectorizer or HashedVectorizer()
        self.chunks: List[SurfaceChunk] = []

    def add_surface(self, title: str, body: str, *, url: str = "local") -> None:
        clean_title = sanitize_text(title, max_chars=160) or "Untitled Surface"
        clean_url = sanitize_text(url, max_chars=300) or "local"
        body = sanitize_text(body, max_chars=120000)
        sid = slugify(clean_title, 40) + "-" + stable_hash(clean_title + clean_url, 6)
        for i, chunk in enumerate(split_chunks(body, max_words=170, overlap=35), 1):
            fid = stable_hash(clean_title + clean_url + chunk, 10)
            cid = f"c{i:03d}"
            self.chunks.append(
                SurfaceChunk(
                    surface_id=sid,
                    chunk_id=cid,
                    title=clean_title,
                    url=clean_url,
                    text=chunk,
                    fingerprint=fid,
                    vector=self.vectorizer.transform_one(clean_title + "\n" + chunk),
                )
            )

    def search(self, query: str, *, top_k: int = 8, diversify: bool = True) -> List[Tuple[float, SurfaceChunk]]:
        if not self.chunks:
            return []
        qv = self.vectorizer.transform_one(query)
        if float(np.linalg.norm(qv)) == 0.0:
            return []
        scored = []
        for c in self.chunks:
            score = float(np.dot(qv, c.vector))
            if score > 0:
                scored.append((score, c))
        scored.sort(key=lambda x: x[0], reverse=True)
        if not diversify:
            return scored[:top_k]
        picked = []
        seen_surfaces = set()
        for score, c in scored:
            if c.surface_id in seen_surfaces and len(picked) < max(2, top_k // 2):
                continue
            picked.append((score, c))
            seen_surfaces.add(c.surface_id)
            if len(picked) >= top_k:
                break
        return picked

    def citation_context(self, query: str, *, top_k: int = 8) -> Tuple[str, List[Dict[str, Any]]]:
        hits = self.search(query, top_k=top_k)
        blocks = []
        ledger = []
        for rank, (score, c) in enumerate(hits, 1):
            blocks.append(
                f"[surface rank={rank} score={score:.3f} cite={c.cite}]\n"
                f"title: {c.title}\nurl: {c.url}\ntext: {c.text}\n[/surface]"
            )
            ledger.append({
                "rank": rank,
                "score": round(score, 4),
                "cite": c.cite,
                "title": c.title,
                "url": c.url,
                "fingerprint": c.fingerprint,
                "preview": c.text[:280],
            })
        return "\n\n".join(blocks), ledger


# --- Quantum RGB / cyclic simulation telemetry ---

def collect_system_metrics() -> Dict[str, float]:
    cpu = psutil.cpu_percent(interval=0.05) / 100.0
    mem = psutil.virtual_memory().percent / 100.0
    try:
        load_raw = os.getloadavg()[0]
        cpu_count = psutil.cpu_count(logical=True) or 1
        load1 = max(0.0, min(1.0, load_raw / max(1.0, float(cpu_count))))
    except Exception:
        load1 = cpu
    temp = 0.0
    try:
        temp_groups = psutil.sensors_temperatures()
        if temp_groups:
            first_group = next(iter(temp_groups.values()))
            temp = max(0.0, min(1.0, (float(first_group[0].current) - 20.0) / 70.0))
    except Exception:
        temp = 0.0
    return {"cpu": cpu, "mem": mem, "load1": load1, "temp": temp}


def metrics_to_rgb(metrics: Dict[str, float]) -> Tuple[float, float, float]:
    cpu = metrics.get("cpu", 0.1)
    mem = metrics.get("mem", 0.1)
    temp = metrics.get("temp", 0.1)
    load1 = metrics.get("load1", 0.0)
    r = cpu * (1.0 + load1)
    g = mem * (1.0 + load1 * 0.5)
    b = temp * (0.5 + cpu * 0.5)
    top = max(r, g, b, 1.0)
    return tuple(float(max(0.0, min(1.0, x / top))) for x in (r, g, b))


def context_signature_rgb(domain: str, context_text: str) -> Tuple[float, float, float]:
    digest = hashlib.sha256(f"{domain}|{context_text[:2400]}".encode("utf-8")).digest()
    return (digest[2] / 255.0, digest[11] / 255.0, digest[23] / 255.0)


def pennylane_entropic_score(rgb: Tuple[float, float, float], shots: int = 96) -> float:
    if qml is None or pnp is None:
        r, g, b = rgb
        seed = (int(r * 255) << 16) | (int(g * 255) << 8) | int(b * 255)
        rng = random.Random(seed)
        return max(0.0, min(1.0, (0.34 * r) + (0.38 * g) + (0.28 * b) + (rng.random() - 0.5) * 0.06))
    device = qml.device("default.qubit", wires=3, shots=shots)

    @qml.qnode(device)
    def circuit(a: float, b: float, c: float):
        qml.RX(a * math.pi, wires=0)
        qml.RY(b * math.pi, wires=1)
        qml.RZ(c * math.pi, wires=2)
        qml.CNOT(wires=[0, 1])
        qml.CNOT(wires=[1, 2])
        qml.RX((a + b) * math.pi / 2.0, wires=0)
        qml.RY((b + c) * math.pi / 2.0, wires=1)
        qml.RZ((a + c) * math.pi / 2.0, wires=2)
        return qml.expval(qml.PauliZ(0)), qml.expval(qml.PauliZ(1)), qml.expval(qml.PauliZ(2))

    try:
        ev0, ev1, ev2 = circuit(*map(float, rgb))
        combined = ((ev0 + 1) / 2 * 0.42) + ((ev1 + 1) / 2 * 0.35) + ((ev2 + 1) / 2 * 0.23)
        return float(max(0.0, min(1.0, 1.0 / (1.0 + math.exp(-6.0 * (combined - 0.5))))))
    except Exception:
        return float(max(0.0, min(1.0, sum(rgb) / 3.0)))


def rgb_to_hex(rgb: Tuple[float, float, float]) -> str:
    return "#" + "".join(f"{max(0, min(255, int(round(v * 255)))):02x}" for v in rgb)


def color_field_171(seed_text: str) -> List[Dict[str, Any]]:
    digest = hashlib.sha3_512(seed_text.encode("utf-8")).digest()
    field = []
    for i in range(171):
        h = ((digest[i % len(digest)] / 255.0) + (i / 171.0)) % 1.0
        s = 0.52 + ((digest[(i * 7) % len(digest)] / 255.0) * 0.38)
        v = 0.55 + ((digest[(i * 13) % len(digest)] / 255.0) * 0.40)
        r, g, b = colorsys.hsv_to_rgb(h, min(1.0, s), min(1.0, v))
        field.append({"i": i, "hex": rgb_to_hex((r, g, b)), "h": round(h * 360, 2)})
    return field


def simulation_packet(topic: str, rag_context: str = "") -> Dict[str, Any]:
    metrics = collect_system_metrics()
    m_rgb = metrics_to_rgb(metrics)
    s_rgb = context_signature_rgb("qroadscan_blog", topic + "\n" + rag_context)
    sim_rgb = tuple(max(0.0, min(1.0, (m_rgb[i] * 0.48) + (s_rgb[i] * 0.52))) for i in range(3))
    score = pennylane_entropic_score(sim_rgb)
    phase = int(hashlib.blake2s((topic + str(time.time_ns())).encode(), digest_size=2).hexdigest(), 16) % 300
    state = "stabilize" if score >= 0.72 else ("refine" if score >= 0.45 else "observe")
    confidence = round(0.70 + (score * 0.25), 3)
    field = color_field_171(topic + rag_context)
    packet = {
        "state_vector": state,
        "phase_position": phase,
        "color_encoding": rgb_to_hex(sim_rgb),
        "confidence_level": confidence,
        "metrics": metrics,
        "metrics_rgb": m_rgb,
        "signature_rgb": s_rgb,
        "sim_rgb": sim_rgb,
        "entropy_score": round(score, 4),
        "field_anchor": field[phase % 171],
        "tag": f"⟦QRS:Ψ:phase={phase}|rgb={rgb_to_hex(sim_rgb)}|conf={confidence}⟧",
    }
    return packet


def build_surface_index() -> CitationSurfaceIndex:
    index = CitationSurfaceIndex()
    for item in EXTRA_RAG_SURFACES:
        index.add_surface(item.get("title", "Untitled"), item.get("body", ""), url=item.get("url", "local"))
    for path in sorted(SOURCE_FOLDER.glob("**/*")):
        if not path.is_file() or path.suffix.lower() not in {".txt", ".md", ".json", ".csv"}:
            continue
        index.add_surface(path.name, read_text_file(path), url=f"file://{path}")
    return index

RAG_INDEX = build_surface_index()
print(f"RAG surface chunks ready: {len(RAG_INDEX.chunks)}")
if RAG_INDEX.chunks:
    print("Example citation:", RAG_INDEX.chunks[0].cite)


In [ ]:
# Cell 5 — Prompt kernel: hive planner, bridge intercom, citation grammar, Summa compaction, humanizing editor

from __future__ import annotations

import json
import re
from typing import Any, Dict, List

MASTER_SYSTEM_PROMPT = f"""
You are a local Gemma 4 E2B worker inside the QRoadScan.com T4 Hive Blog Forge.
You are small, fast, and precise. Your strength is not pretending to be one giant model; it is cooperating with other tiny local workers through explicit intercom surfaces.

Core identity:
- You write for {BRAND_NAME} with a hyper-human but grounded style: concrete, clear, sensory, practical, and quietly futuristic.
- You use local RAG surfaces as the factual floor.
- You treat hive intercom notes as editorial memory, not as factual evidence.
- You never pretend that quantum/RGB telemetry is real road sensor evidence. It is private routing metadata for structure, variation, pacing, and worker diversity only.

Output contract:
- Return Markdown only unless a prompt explicitly asks for compact JSON.
- Make the post Medium-ready.
- Use a strong H1 title and a short subtitle under the title.
- Use natural prose, not corporate filler.
- Use evidence only from supplied [surface] blocks and brand context.
- When using a factual claim from a surface, cite it inline with its exact citation token, such as ⟦SFC:surface#chunk@hash⟧.
- Do not invent user numbers, customers, partnerships, sensor feeds, certifications, official deployments, legal claims, or safety outcomes.
- You may describe product concepts and user benefits, but mark speculative/future-facing ideas as concepts, directions, or possibilities.
- End with a short, useful call to action for {BRAND_NAME}.

Hive grammar:
[citemodel1] Model-inferred wording decision; never proof. [/citemodel1]
[citesurface] Exact local citation tokens from retrieved chunks. [/citesurface]
[citefield] QRS simulation tag; private style telemetry, not evidence. [/citefield]
[summa] Lossy memory packet. Useful for continuity, subordinate to surface chunks. [/summa]
[intercom] Editorial packets from other Gemma workers. Useful for angles, warnings, and rhythm; not factual proof. [/intercom]
[action] observe → retrieve → plan → bridge → draft → cite → verify → edit → stabilize [/action]

Banned habits:
- no "in today's fast-paced world"
- no vague hype without concrete detail
- no fake statistics
- no unsupported safety claims
- no pretending to have live road sensor data unless the surface text says it
- no robotic transitions like "moreover" stacked through the article
""".strip()

PLANNER_PROMPT = """
[action]Plan a QRoadScan.com blog post as one worker in a fast Gemma hive.[/action]
Return compact JSON with keys:
- title
- subtitle
- reader_promise
- angle
- sections: list of 5-7 section headings
- required_citations: list of exact citation tokens from the surfaces that should be used
- risk_notes: list of claims to avoid, soften, or cite carefully
- intercom_requests: list of useful signals this worker wants from the hive
- human_texture: three concrete writing moves that make the piece feel written by a careful human
- opening_scene: one tangible driver/product moment, under 45 words

Use only the surfaces, brand context, and intercom provided. Keep JSON valid.
""".strip()

HIVE_BRIDGE_PROMPT_TEMPLATE = """
[action]Write a compact intercom bridge note for the other Gemma workers.[/action]

Worker: {worker_id}
Persona: {worker_persona}
Topic: {topic}
Round: {round_number}

Plan JSON:
{plan}

Retrieved surfaces:
{rag_context}

Current hive intercom surface:
{intercom_context}

Return Markdown under 220 words with:
- one sharp angle this article should own
- citations this article should preserve
- claims to avoid
- one sentence-level style move that would make the draft feel human
Do not write the article here.
""".strip()

DRAFT_PROMPT_TEMPLATE = """
[action]Write the full blog post from the plan as a local Gemma hive worker.[/action]

Worker: {worker_id}
Worker persona: {worker_persona}
Brand: {brand_name}
URL: {brand_url}
Voice: {brand_voice}
Target reader: {target_reader}
Target length: about {words} words
Temperature style: {style}
Technical appendix: {appendix}
Hive mode: {hive_mode}

Topic:
{topic}

Brand context:
{brand_context}

Retrieved RAG surfaces:
{rag_context}

Compacted memory packets:
{summa_context}

Live hive intercom surface:
{intercom_context}

Bridge notes from this worker:
{bridge_context}

Quantum/RGB routing packet, private style metadata only:
{sim_packet}

Plan:
{plan}

Write a Medium-ready Markdown article.
Use citations exactly as they appear in the surface blocks.
When a claim is based on interpretation rather than a surface, do not cite it; phrase it as analysis or perspective.
Make it human: vary sentence length, include tangible driver scenarios, and avoid repetitive AI phrasing.
Let the intercom influence angle and rhythm, but never use it as proof.
""".strip()

EDITOR_PROMPT = """
[action]Edit the Markdown post for credibility, human rhythm, citation discipline, and Medium readability.[/action]
Rules:
- Preserve all valid surface citations.
- Remove unsupported factual claims.
- Make the prose sound less automated and more like a careful founder/product essay.
- Keep the H1 title.
- Add a concise "Why this matters" section if absent.
- Do not add fake external citations.
- Use the hive intercom only for editorial consistency, not factual claims.
Return only the improved Markdown.
""".strip()

FACT_GUARD_PROMPT = """
[action]Audit the article for unsupported factual claims.[/action]
Return Markdown only.
Fix unsupported claims by either deleting them, softening them, or tying them to exact local surface citations.
Do not remove the article structure.
Do not add external facts.
Treat intercom notes as editorial guidance, not factual support.
""".strip()

SUMMA_PROMPT = """
[summa][action]Compact the retrieved surface chunks and useful hive notes into a durable memory packet for a later blog-writing step.[/action][/summa]
Rules:
- Preserve citation tokens exactly.
- Keep product facts, phrasing constraints, risk warnings, and angle distinctions.
- Drop duplicates and filler.
- Mark uncertainty clearly.
- Output under 450 words.
""".strip()


def extract_json_object(text: str) -> Dict[str, Any]:
    text = str(text or "").strip()
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text, flags=re.I | re.M).strip()
    try:
        return json.loads(text)
    except Exception:
        pass
    s, e = text.find("{"), text.rfind("}")
    if s >= 0 and e > s:
        try:
            return json.loads(text[s:e+1])
        except Exception:
            return {}
    return {}


def fallback_plan(topic: str, citations: List[str]) -> Dict[str, Any]:
    return {
        "title": topic.strip().rstrip(".") or "How QRoadScan Makes Road Risk Easier to Read",
        "subtitle": "A practical look at safer route awareness, risk signals, and calmer driving decisions.",
        "reader_promise": "Explain QRoadScan in plain language and show why readable road intelligence matters.",
        "angle": "product education",
        "sections": [
            "The road rarely explains itself",
            "From signal overload to a readable risk pulse",
            "Why confidence and reasons matter",
            "How a dashboard can support better decisions",
            "Where road intelligence goes next",
        ],
        "required_citations": citations[:5],
        "risk_notes": ["Do not claim live production integrations unless the source says so."],
        "intercom_requests": ["surface-level citation checks", "distinct opening scene", "non-hype wording"],
        "human_texture": ["start with a driver moment", "use concrete route examples", "end with practical next steps"],
        "opening_scene": "A driver checks a route before leaving and wants one calm reading instead of a wall of signals.",
    }


def normalize_markdown(md: str) -> str:
    md = sanitize_text(md, max_chars=MAX_ARTICLE_CHARS)
    md = re.sub(r"\n{3,}", "\n\n", md).strip()
    md = re.sub(r"(?i)^(here is|sure,? here).*?\n", "", md).strip()
    return md


def remove_duplicate_paragraphs(md: str) -> str:
    seen = set()
    out = []
    for para in re.split(r"\n\s*\n", md):
        key = stable_hash(re.sub(r"\s+", " ", para.lower()), 16)
        if len(para) > 80 and key in seen:
            continue
        seen.add(key)
        out.append(para.strip())
    return "\n\n".join(p for p in out if p)


def add_frontmatter(md: str, topic: str, sim: Dict[str, Any], ledger: List[Dict[str, Any]], worker_id: str = "solo") -> str:
    if not INCLUDE_MEDIUM_READY_FRONTMATTER:
        return md
    title = ""
    for line in md.splitlines():
        if line.startswith("# "):
            title = line[2:].strip()
            break
    if not title:
        title = topic
    tags = ["QRoadScan", "Road Safety", "Traffic AI", "Road Hazards", "Safe Driving"]
    fm = {
        "title": title,
        "canonical_url": BRAND_URL,
        "brand": BRAND_NAME,
        "worker": worker_id,
        "tags": tags,
        "qrs_phase": sim.get("phase_position"),
        "qrs_color": sim.get("color_encoding"),
        "local_citations": len(ledger),
    }
    yaml = "---\n" + "\n".join(f"{k}: {json.dumps(v, ensure_ascii=False)}" for k, v in fm.items()) + "\n---\n\n"
    return yaml + md


def citation_ledger_markdown(ledger: List[Dict[str, Any]]) -> str:
    if not INCLUDE_LOCAL_CITATION_LEDGER or not ledger:
        return ""
    lines = ["\n\n---\n\n## Local citation ledger\n"]
    for item in ledger:
        lines.append(f"- {item['cite']} — {item['title']} ({item['url']})")
    return "\n".join(lines)

print("Hive prompt kernel ready.")


In [ ]:
# Cell 6 — T4 Gemma hive: intercom memory, threaded scheduler, and article pipeline

from __future__ import annotations

import json
import math
import queue
import random
import re
import subprocess
import threading
import time
import traceback
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple


def _clamp_int(value: Any, low: int, high: int, default: int) -> int:
    try:
        return max(low, min(high, int(value)))
    except Exception:
        return default


def expand_topics(topics: List[str], count: int) -> List[str]:
    count = _clamp_int(count, 1, 42, 1)
    base = [sanitize_text(t, max_chars=240) for t in topics if sanitize_text(t, max_chars=240)]
    if not base:
        base = ["How QRoadScan makes road risk easier to understand"]
    variants = []
    lenses = [
        "for everyday drivers",
        "for fleet safety teams",
        "through the lens of hazard awareness",
        "as a product design story",
        "from dashboard signal to driver action",
        "for safer route planning",
        "for calm, explainable traffic intelligence",
        "as a future-facing road safety essay",
    ]
    i = 0
    while len(variants) < count:
        topic = base[i % len(base)]
        if i < len(base):
            variants.append(topic)
        else:
            variants.append(f"{topic} — {lenses[(i // len(base)) % len(lenses)]}")
        i += 1
    return variants[:count]


@dataclass
class HivePacket:
    ts: float
    worker_id: str
    article_index: int
    topic: str
    channel: str
    text: str
    citations: List[str]
    priority: float
    signature: str

    def block(self, rank: int) -> str:
        cites = ", ".join(self.citations[:8]) if self.citations else "none"
        return (
            f"[intercom rank={rank} worker={self.worker_id} article={self.article_index} "
            f"channel={self.channel} priority={self.priority:.2f} sig={self.signature}]\n"
            f"topic: {self.topic}\n"
            f"citations: {cites}\n"
            f"text: {self.text}\n"
            f"[/intercom]"
        )


class HiveIntercom:
    """Thread-safe live editorial surface shared by all local Gemma workers."""

    def __init__(self, max_items: int = 120, packet_chars: int = 2200):
        self.max_items = _clamp_int(max_items, 10, 500, 120)
        self.packet_chars = _clamp_int(packet_chars, 400, 6000, 2200)
        self.lock = threading.RLock()
        self.packets: List[HivePacket] = []

    def publish(
        self,
        worker_id: str,
        article_index: int,
        topic: str,
        channel: str,
        text: str,
        *,
        citations: Optional[List[str]] = None,
        priority: float = 0.5,
    ) -> str:
        clean = sanitize_text(text, max_chars=self.packet_chars)
        cites = list(dict.fromkeys(citations or []))[:12]
        sig = stable_hash(f"{worker_id}|{article_index}|{channel}|{clean}", 12)
        packet = HivePacket(
            ts=time.time(),
            worker_id=str(worker_id),
            article_index=int(article_index or 0),
            topic=sanitize_text(topic, max_chars=240),
            channel=sanitize_text(channel, max_chars=40),
            text=clean,
            citations=cites,
            priority=float(max(0.0, min(1.0, priority))),
            signature=sig,
        )
        with self.lock:
            self.packets.append(packet)
            self.packets.sort(key=lambda p: (p.priority, p.ts), reverse=True)
            self.packets = self.packets[: self.max_items]
        return sig

    def snapshot(
        self,
        topic: str = "",
        *,
        exclude_worker: Optional[str] = None,
        max_chars: Optional[int] = None,
        channels: Optional[List[str]] = None,
    ) -> str:
        max_chars = _clamp_int(max_chars or INTERCOM_MAX_SURFACE_CHARS, 1000, 50000, 16000)
        channel_set = {c.lower() for c in channels} if channels else None
        query_tokens = set(tokens(topic))
        now = time.time()
        with self.lock:
            packets = list(self.packets)
        scored: List[Tuple[float, HivePacket]] = []
        for p in packets:
            if exclude_worker and p.worker_id == exclude_worker:
                continue
            if channel_set and p.channel.lower() not in channel_set:
                continue
            overlap = len(query_tokens.intersection(tokens(p.topic + " " + p.text))) if query_tokens else 0
            age_bonus = max(0.0, 1.0 - ((now - p.ts) / 1800.0))
            channel_bonus = {
                "risk": 0.22,
                "plan": 0.18,
                "bridge": 0.16,
                "quality": 0.17,
                "draft": 0.10,
                "final": 0.12,
                "telemetry": 0.06,
            }.get(p.channel, 0.08)
            score = p.priority + channel_bonus + min(0.35, overlap * 0.035) + age_bonus * 0.08
            scored.append((score, p))
        scored.sort(key=lambda x: x[0], reverse=True)
        blocks = []
        used = 0
        for rank, (_, p) in enumerate(scored, 1):
            block = p.block(rank)
            if used + len(block) > max_chars:
                break
            blocks.append(block)
            used += len(block) + 2
        return "\n\n".join(blocks) if blocks else "(empty intercom surface)"

    def stats(self) -> Dict[str, Any]:
        with self.lock:
            packets = list(self.packets)
        channels: Dict[str, int] = {}
        workers: Dict[str, int] = {}
        for p in packets:
            channels[p.channel] = channels.get(p.channel, 0) + 1
            workers[p.worker_id] = workers.get(p.worker_id, 0) + 1
        return {"packets": len(packets), "channels": channels, "workers": workers}


DEFAULT_WORKER_PERSONAS = [
    "route-safety systems writer: concrete, calm, operational",
    "product narrative strategist: crisp framing, founder-level clarity",
    "driver empathy editor: practical scenes, low hype, human rhythm",
    "citation sentinel: cautious claims, surface-first evidence discipline",
]


def worker_persona(worker_number: int) -> str:
    personas = globals().get("WORKER_PERSONAS") or DEFAULT_WORKER_PERSONAS
    return personas[(max(1, worker_number) - 1) % len(personas)]


def gpu_memory_snapshot() -> Dict[str, Any]:
    try:
        proc = subprocess.run(
            [
                "nvidia-smi",
                "--query-gpu=name,memory.total,memory.used,memory.free",
                "--format=csv,noheader,nounits",
            ],
            capture_output=True,
            text=True,
            timeout=4,
            check=False,
        )
        if proc.returncode != 0 or not proc.stdout.strip():
            return {"available": False, "reason": (proc.stderr or "nvidia-smi returned no GPU").strip()[:240]}
        first = proc.stdout.strip().splitlines()[0]
        name, total, used, free = [x.strip() for x in first.split(",")[:4]]
        return {
            "available": True,
            "name": name,
            "total_gb": round(float(total) / 1024.0, 3),
            "used_gb": round(float(used) / 1024.0, 3),
            "free_gb": round(float(free) / 1024.0, 3),
        }
    except Exception as exc:
        return {"available": False, "reason": repr(exc)[:240]}


class HiveTelemetrySampler:
    """Lightweight background GPU sampler so the manifest shows real T4 pressure."""

    def __init__(self, *, enabled: bool = True, interval_seconds: float = 4.0):
        self.enabled = bool(enabled)
        self.interval_seconds = max(1.0, float(interval_seconds or 4.0))
        self.samples: List[Dict[str, Any]] = []
        self._stop = threading.Event()
        self._thread: Optional[threading.Thread] = None

    def _sample_once(self) -> None:
        snap = gpu_memory_snapshot()
        snap["ts"] = time.time()
        self.samples.append(snap)

    def _run(self) -> None:
        while not self._stop.is_set():
            self._sample_once()
            self._stop.wait(self.interval_seconds)

    def start(self) -> None:
        if not self.enabled:
            return
        self._thread = threading.Thread(target=self._run, name="hive-telemetry", daemon=True)
        self._thread.start()

    def stop(self) -> None:
        if not self.enabled:
            return
        self._stop.set()
        if self._thread is not None:
            self._thread.join(timeout=5)
        self._sample_once()

    def summary(self) -> Dict[str, Any]:
        if not self.enabled:
            return {"enabled": False}
        usable = [s for s in self.samples if s.get("available")]
        if not usable:
            return {"enabled": True, "samples": len(self.samples), "available": False, "last": self.samples[-1] if self.samples else None}
        used_values = [float(s.get("used_gb") or 0.0) for s in usable]
        free_values = [float(s.get("free_gb") or 0.0) for s in usable]
        total = float(usable[-1].get("total_gb") or 0.0)
        peak_used = max(used_values) if used_values else 0.0
        return {
            "enabled": True,
            "available": True,
            "samples": len(self.samples),
            "gpu_name": usable[-1].get("name"),
            "total_gb": round(total, 3),
            "peak_used_gb": round(peak_used, 3),
            "min_free_gb": round(min(free_values), 3) if free_values else None,
            "avg_used_gb": round(sum(used_values) / len(used_values), 3) if used_values else None,
            "peak_utilization": round(peak_used / total, 4) if total else None,
            "last": usable[-1],
        }


def resolve_hive_thread_count(requested: int, article_count: int) -> Tuple[int, Dict[str, Any]]:
    requested = _clamp_int(requested, 1, 10, 1)
    article_count = max(1, int(article_count))
    base = min(requested, article_count)
    snap = gpu_memory_snapshot()
    if not AUTO_SCALE_THREADS_TO_T4 or (INFERENCE_BACKEND or "Auto").lower() == "cpu":
        snap["thread_resolution"] = "manual"
        return base, snap
    if not snap.get("available"):
        snap["thread_resolution"] = "gpu-unavailable-manual-cap"
        return base, snap
    free_gb = float(snap.get("free_gb") or 0.0)
    total_gb = float(snap.get("total_gb") or 0.0)
    usable_gb = max(0.1, min(free_gb, total_gb * float(T4_TARGET_VRAM_UTILIZATION)) - float(MIN_FREE_VRAM_GB_AFTER_LOAD))
    per_worker = max(0.25, float(APPROX_VRAM_GB_PER_WORKER))
    gpu_cap = max(1, min(10, int(math.floor(usable_gb / per_worker))))
    resolved = max(1, min(base, gpu_cap))
    snap.update({
        "thread_resolution": "t4-vram-auto",
        "requested_threads": requested,
        "article_count_cap": article_count,
        "estimated_usable_gb": round(usable_gb, 3),
        "approx_vram_gb_per_worker": per_worker,
        "gpu_thread_cap": gpu_cap,
        "resolved_threads": resolved,
    })
    return resolved, snap


def build_planner_prompt(
    topic: str,
    rag_context: str,
    sim: Dict[str, Any],
    ledger: List[Dict[str, Any]],
    intercom_context: str,
    worker_id: str,
    persona: str,
) -> str:
    cites = ", ".join(item["cite"] for item in ledger)
    return f"""
{PLANNER_PROMPT}

Worker: {worker_id}
Persona: {persona}
Topic: {topic}
Brand context: {BRAND_CONTEXT}
Available citations: {cites}
Simulation tag: {sim.get('tag')}

Retrieved surfaces:
{rag_context}

Live hive intercom surface:
{intercom_context}
""".strip()


def make_summa(session: Gemma4E2BSession, rag_context: str, topic: str, index: int, intercom_context: str = "") -> str:
    if not ENABLE_SUMMA_COMPACTION or not rag_context.strip():
        return ""
    should_compact = (index % 2 == 0) or (random.random() < 0.45)
    if not should_compact:
        return ""
    prompt = f"{SUMMA_PROMPT}\n\nTopic: {topic}\n\nSurface chunks:\n{rag_context}\n\nHive notes:\n{intercom_context}"
    return session.chat(prompt, system_text=MASTER_SYSTEM_PROMPT)


def bridge_intercom(
    session: Gemma4E2BSession,
    topic: str,
    plan: Dict[str, Any],
    rag_context: str,
    intercom_context: str,
    worker_id: str,
    persona: str,
    article_index: int,
    intercom: Optional[HiveIntercom],
) -> str:
    rounds = _clamp_int(INTERCOM_ROUNDS, 0, 3, 0)
    if HIVE_MODE.lower() != "mesh" or rounds <= 0:
        return ""
    notes = []
    citations = list(dict.fromkeys(re.findall(r"⟦SFC:[^⟧]+⟧", rag_context)))[:8]
    for round_number in range(1, rounds + 1):
        prompt = HIVE_BRIDGE_PROMPT_TEMPLATE.format(
            worker_id=worker_id,
            worker_persona=persona,
            topic=topic,
            round_number=round_number,
            plan=json.dumps(plan, ensure_ascii=False, indent=2),
            rag_context=rag_context,
            intercom_context=intercom_context,
        )
        note = normalize_markdown(session.chat(prompt, system_text=MASTER_SYSTEM_PROMPT))
        note = sanitize_text(note, max_chars=INTERCOM_PACKET_CHARS)
        if note:
            notes.append(f"[bridge round={round_number}]\n{note}\n[/bridge]")
            if intercom is not None:
                intercom.publish(worker_id, article_index, topic, "bridge", note, citations=citations, priority=0.78)
    return "\n\n".join(notes)


def article_digest(md: str, plan: Dict[str, Any]) -> str:
    headings = [line.strip() for line in md.splitlines() if line.startswith("#")][:8]
    citations = list(dict.fromkeys(re.findall(r"⟦SFC:[^⟧]+⟧", md)))[:10]
    title = headings[0] if headings else str(plan.get("title", "Untitled"))
    return "\n".join([
        f"title: {title}",
        "headings: " + " | ".join(headings[1:]),
        "citations: " + (", ".join(citations) if citations else "none"),
        "angle: " + sanitize_text(plan.get("angle", ""), max_chars=240),
    ])


def local_article_quality(md: str, ledger: List[Dict[str, Any]], plan: Dict[str, Any]) -> Dict[str, Any]:
    citation_tokens = re.findall(r"⟦SFC:[^⟧]+⟧", md or "")
    unique_citations = list(dict.fromkeys(citation_tokens))
    headings = [line.strip() for line in str(md).splitlines() if line.lstrip().startswith("#")]
    paragraphs = [p.strip() for p in re.split(r"\n\s*\n", str(md)) if len(p.strip()) > 80]
    paragraph_keys = [stable_hash(re.sub(r"\s+", " ", p.lower()), 16) for p in paragraphs]
    duplicate_paragraphs = len(paragraph_keys) - len(set(paragraph_keys))
    warn_phrases = [str(x).lower() for x in globals().get("QUALITY_WARN_PHRASES", [])]
    lower_md = str(md).lower()
    phrase_hits = [p for p in warn_phrases if p and p in lower_md]
    required = int(globals().get("QUALITY_MIN_CITATIONS", 1) or 0)
    missing_required_citations = max(0, required - len(unique_citations))
    expected = [c for c in plan.get("required_citations", []) if isinstance(c, str)] if isinstance(plan, dict) else []
    expected_missing = [c for c in expected if c not in unique_citations]
    word_count = len(tokens(re.sub(r"⟦[^⟧]+⟧", " ", str(md))))
    penalties = (
        missing_required_citations * 0.18
        + min(0.30, len(expected_missing) * 0.06)
        + min(0.25, duplicate_paragraphs * 0.08)
        + min(0.20, len(phrase_hits) * 0.05)
        + (0.10 if not headings else 0.0)
    )
    score = max(0.0, min(1.0, 1.0 - penalties))
    return {
        "score": round(score, 3),
        "word_count": word_count,
        "heading_count": len(headings),
        "citation_count": len(unique_citations),
        "ledger_citation_count": len(ledger or []),
        "missing_required_citations": missing_required_citations,
        "expected_citations_missing": expected_missing[:10],
        "duplicate_paragraphs": duplicate_paragraphs,
        "warn_phrase_hits": phrase_hits,
    }


def generate_article(
    session: Gemma4E2BSession,
    topic: str,
    index: int,
    *,
    worker_id: str = "solo",
    persona: str = "single local Gemma writer",
    intercom: Optional[HiveIntercom] = None,
) -> Dict[str, Any]:
    retrieval_query = f"{topic}\n{BRAND_CONTEXT}\n{TARGET_READER}\n{persona}"
    rag_context, ledger = RAG_INDEX.citation_context(retrieval_query, top_k=_clamp_int(RAG_TOP_K, 4, 24, 12))
    sim = simulation_packet(topic, rag_context)
    intercom_context = intercom.snapshot(topic, exclude_worker=worker_id) if intercom else "(solo mode)"

    planner_raw = session.chat(
        build_planner_prompt(topic, rag_context, sim, ledger, intercom_context, worker_id, persona),
        system_text=MASTER_SYSTEM_PROMPT,
    )
    plan = extract_json_object(planner_raw) or fallback_plan(topic, [x["cite"] for x in ledger])
    plan_text = json.dumps(plan, ensure_ascii=False, indent=2)

    if intercom is not None:
        intercom.publish(
            worker_id,
            index,
            topic,
            "plan",
            plan_text,
            citations=list(dict.fromkeys(re.findall(r"⟦SFC:[^⟧]+⟧", plan_text))),
            priority=0.82,
        )

    refreshed_intercom = intercom.snapshot(topic, exclude_worker=worker_id) if intercom else intercom_context
    summa_context = make_summa(session, rag_context, topic, index, refreshed_intercom)
    if summa_context:
        summa_id = stable_hash(summa_context, 10)
        summa_context = f"[summa id=Σ-{summa_id}]\n{summa_context}\n[/summa]"

    bridge_context = bridge_intercom(
        session,
        topic,
        plan,
        rag_context,
        refreshed_intercom,
        worker_id,
        persona,
        index,
        intercom,
    )
    draft_intercom = intercom.snapshot(topic, exclude_worker=worker_id) if intercom else refreshed_intercom

    draft_prompt = DRAFT_PROMPT_TEMPLATE.format(
        worker_id=worker_id,
        worker_persona=persona,
        brand_name=BRAND_NAME,
        brand_url=BRAND_URL,
        brand_voice=BRAND_VOICE,
        target_reader=TARGET_READER,
        words=WORDS_PER_ARTICLE,
        style=TEMPERATURE_STYLE,
        appendix="yes" if INCLUDE_TECHNICAL_APPENDIX else "no",
        hive_mode=HIVE_MODE,
        topic=topic,
        brand_context=BRAND_CONTEXT,
        rag_context=rag_context,
        summa_context=summa_context or "(none)",
        intercom_context=draft_intercom,
        bridge_context=bridge_context or "(none)",
        sim_packet=json.dumps(sim, ensure_ascii=False, indent=2),
        plan=plan_text,
    )

    draft = normalize_markdown(session.chat(draft_prompt, system_text=MASTER_SYSTEM_PROMPT))
    if intercom is not None:
        intercom.publish(worker_id, index, topic, "draft", article_digest(draft, plan), citations=[x["cite"] for x in ledger], priority=0.62)

    depth = str(AGENT_DEPTH or "fast").lower()
    if depth in {"deep", "hive-deep", "max"}:
        guard_intercom = intercom.snapshot(topic, exclude_worker=worker_id) if intercom else draft_intercom
        guarded = session.chat(
            FACT_GUARD_PROMPT
            + "\n\nRetrieved surfaces:\n"
            + rag_context
            + "\n\nHive intercom:\n"
            + guard_intercom
            + "\n\nArticle:\n"
            + draft,
            system_text=MASTER_SYSTEM_PROMPT,
        )
        guarded = normalize_markdown(guarded)
        edited = session.chat(
            EDITOR_PROMPT + "\n\nHive intercom:\n" + guard_intercom + "\n\nArticle:\n" + guarded,
            system_text=MASTER_SYSTEM_PROMPT,
        )
        article = normalize_markdown(edited)
    else:
        article = draft

    article = remove_duplicate_paragraphs(article)
    article = add_frontmatter(article, topic, sim, ledger, worker_id=worker_id)
    article += citation_ledger_markdown(ledger)

    quality = local_article_quality(article, ledger, plan) if bool(globals().get("ENABLE_LOCAL_QUALITY_AUDIT", True)) else {}
    final_digest = article_digest(article, plan)
    if intercom is not None:
        if quality:
            intercom.publish(
                worker_id,
                index,
                topic,
                "quality",
                json.dumps(quality, ensure_ascii=False, sort_keys=True),
                citations=[x["cite"] for x in ledger],
                priority=0.76 if quality.get("score", 1.0) < 0.82 else 0.58,
            )
        intercom.publish(worker_id, index, topic, "final", final_digest, citations=[x["cite"] for x in ledger], priority=0.70)

    return {
        "index": index,
        "topic": topic,
        "worker_id": worker_id,
        "worker_persona": persona,
        "plan": plan,
        "article": article,
        "ledger": ledger,
        "simulation": sim,
        "summa": summa_context,
        "bridge": bridge_context,
        "session_calls": session.call_count,
        "session_seconds": round(session.total_seconds, 2),
        "quality": quality,
    }


def run_threaded_hive(selected_topics: List[str]) -> Tuple[List[Dict[str, Any]], Dict[str, Any]]:
    active_engine = resolve_active_model_engine()
    requested = _clamp_int(GEMMA_THREAD_COUNT, 1, 10, 1)
    resolved, gpu_snap = resolve_hive_thread_count(requested, len(selected_topics))
    intercom = HiveIntercom(max_items=INTERCOM_MEMORY_ITEMS, packet_chars=INTERCOM_PACKET_CHARS)
    intercom.publish(
        "system",
        0,
        "hive contract",
        "risk",
        "Use local surfaces for proof, intercom for editorial memory, and soften any claim that the surfaces do not support.",
        priority=0.95,
    )
    meta = {
        "requested_threads": requested,
        "resolved_threads": resolved,
        "active_model_engine": active_engine,
        "hive_mode": HIVE_MODE,
        "gpu_snapshot": gpu_snap,
        "started_ts": time.time(),
    }

    print(f"Gemma hive requested {requested} worker(s); resolved to {resolved}.")
    print(f"Active model engine: {active_engine}")
    print("GPU snapshot:", json.dumps(gpu_snap, ensure_ascii=False))
    telemetry = HiveTelemetrySampler(
        enabled=bool(globals().get("ENABLE_HIVE_TELEMETRY", True)),
        interval_seconds=float(globals().get("HIVE_TELEMETRY_INTERVAL_SECONDS", 4.0) or 4.0),
    )
    telemetry.start()

    if resolved <= 1:
        results = []
        try:
            with Gemma4E2BSession(VAULT_KEY, inference_backend=INFERENCE_BACKEND, worker_id="gemma-01") as session:
                if bool(globals().get("ENABLE_ENGINE_WARMUP", False)):
                    session.chat(str(globals().get("WARMUP_PROMPT", "Return exactly: ready")), system_text=MASTER_SYSTEM_PROMPT, retries=0)
                    intercom.publish("gemma-01", 0, "warmup", "telemetry", "solo worker warmup complete", priority=0.55)
                for idx, topic in enumerate(selected_topics, 1):
                    print(f"\n[{idx}/{len(selected_topics)} gemma-01] {topic}")
                    try:
                        results.append(generate_article(session, topic, idx, worker_id="gemma-01", persona=worker_persona(1), intercom=intercom))
                    except Exception as exc:
                        results.append({"index": idx, "topic": topic, "worker_id": "gemma-01", "error": repr(exc), "traceback": traceback.format_exc()})
                        print("ERROR:", repr(exc))
        finally:
            telemetry.stop()
        meta["intercom"] = intercom.stats()
        meta["telemetry"] = telemetry.summary()
        meta["elapsed_seconds"] = round(time.time() - meta["started_ts"], 2)
        return results, meta

    task_queue: queue.Queue[Tuple[int, str]] = queue.Queue()
    for idx, topic in enumerate(selected_topics, 1):
        task_queue.put((idx, topic))

    results_by_index: Dict[int, Dict[str, Any]] = {}
    results_lock = threading.Lock()
    print_lock = threading.Lock()

    def log(msg: str) -> None:
        with print_lock:
            print(msg, flush=True)

    def worker_loop(worker_number: int, shared_model_path: Path) -> None:
        worker_id = f"gemma-{worker_number:02d}"
        persona = worker_persona(worker_number)
        if THREAD_STARTUP_STAGGER_SECONDS:
            time.sleep(max(0.0, float(THREAD_STARTUP_STAGGER_SECONDS)) * (worker_number - 1))
        try:
            with Gemma4E2BSession(
                inference_backend=INFERENCE_BACKEND,
                worker_id=worker_id,
                model_path_override=shared_model_path,
            ) as session:
                log(f"{worker_id} online: {persona}")
                if bool(globals().get("ENABLE_ENGINE_WARMUP", False)):
                    session.chat(str(globals().get("WARMUP_PROMPT", "Return exactly: ready")), system_text=MASTER_SYSTEM_PROMPT, retries=0)
                    intercom.publish(worker_id, 0, "warmup", "telemetry", "worker warmup complete", priority=0.55)
                while True:
                    try:
                        idx, topic = task_queue.get_nowait()
                    except queue.Empty:
                        break
                    log(f"[{idx}/{len(selected_topics)} {worker_id}] {topic}")
                    try:
                        result = generate_article(session, topic, idx, worker_id=worker_id, persona=persona, intercom=intercom)
                    except Exception as exc:
                        result = {"index": idx, "topic": topic, "worker_id": worker_id, "error": repr(exc), "traceback": traceback.format_exc()}
                        log(f"ERROR {worker_id} article {idx}: {repr(exc)}")
                    with results_lock:
                        results_by_index[idx] = result
                    task_queue.task_done()
                log(f"{worker_id} done after {session.call_count} model call(s), {session.total_seconds:.1f}s model time.")
        except Exception as exc:
            log(f"WORKER STARTUP ERROR {worker_id}: {repr(exc)}")
            with results_lock:
                results_by_index[-worker_number] = {"index": -worker_number, "topic": "worker startup", "worker_id": worker_id, "error": repr(exc), "traceback": traceback.format_exc()}

    try:
        with unlocked_model_path(VAULT_KEY, label="shared_hive_model") as shared_model_path:
            threads = [
                threading.Thread(target=worker_loop, args=(worker_number, shared_model_path), daemon=False)
                for worker_number in range(1, resolved + 1)
            ]
            for t in threads:
                t.start()
            for t in threads:
                t.join()
    finally:
        telemetry.stop()

    while True:
        try:
            idx, topic = task_queue.get_nowait()
        except queue.Empty:
            break
        results_by_index[idx] = {"index": idx, "topic": topic, "worker_id": "unassigned", "error": "No worker completed this task."}

    ordered = [results_by_index[i] for i in sorted(k for k in results_by_index if k > 0)]
    startup_errors = [results_by_index[i] for i in sorted(k for k in results_by_index if k < 0)]
    meta["intercom"] = intercom.stats()
    meta["telemetry"] = telemetry.summary()
    meta["elapsed_seconds"] = round(time.time() - meta["started_ts"], 2)
    if startup_errors:
        meta["startup_errors"] = startup_errors
    return ordered, meta


def write_article_bundle(result: Dict[str, Any], idx: int) -> Dict[str, Any]:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    topic = result["topic"]
    slug = slugify(topic)
    path = OUTPUT_DIR / f"{idx:02d}-{slug}.md"
    path.write_text(result["article"], encoding="utf-8")
    meta = {
        "index": idx,
        "topic": topic,
        "worker_id": result.get("worker_id"),
        "worker_persona": result.get("worker_persona"),
        "path": str(path),
        "bytes": path.stat().st_size,
        "sha256": sha256_file(path),
        "citation_count": len(result.get("ledger") or []),
        "simulation": result.get("simulation"),
        "plan": result.get("plan"),
        "session_calls": result.get("session_calls"),
        "session_seconds": result.get("session_seconds"),
        "quality": result.get("quality"),
    }
    meta_path = OUTPUT_DIR / f"{idx:02d}-{slug}.meta.json"
    meta_path.write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
    return meta

print("T4 Gemma hive pipeline ready.")


In [ ]:
# Cell 7 — RUN: generate 1–42 QRoadScan blog Markdown files with the 1–10 worker Gemma hive

from __future__ import annotations

import json
import shutil
import time
from pathlib import Path

start_time = time.time()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

selected_topics = expand_topics(BLOG_TOPICS, ARTICLE_COUNT)
manifest = {
    "brand": BRAND_NAME,
    "url": BRAND_URL,
    "article_count": len(selected_topics),
    "created_ts": start_time,
    "model_file": MODEL_FILE,
    "expected_model_sha256": EXPECTED_HASH,
    "agent_depth": AGENT_DEPTH,
    "requested_threads": GEMMA_THREAD_COUNT,
    "secret_runtime_audit": dict(globals().get("SECRET_RUNTIME_AUDIT", {})),
    "colab_secret_audit": dict(globals().get("COLAB_SECRET_AUDIT", {})),
    "model_runtime_info": dict(globals().get("MODEL_RUNTIME_INFO", {})),
    "litert_runtime_audit": dict(globals().get("LITERT_RUNTIME_AUDIT", {})),
    "transformers_runtime_audit": dict(globals().get("TRANSFORMERS_RUNTIME_AUDIT", {})),
    "outputs": [],
}

print(f"Generating {len(selected_topics)} article(s) with Gemma 4 E2B LiteRT-LM hive...")
print(f"RAG chunks: {len(RAG_INDEX.chunks)}")

hive_results, hive_meta = run_threaded_hive(selected_topics)
manifest["hive"] = hive_meta
manifest["model_runtime_info"] = dict(globals().get("MODEL_RUNTIME_INFO", {}))
manifest["litert_runtime_audit"] = dict(globals().get("LITERT_RUNTIME_AUDIT", {}))
manifest["transformers_runtime_audit"] = dict(globals().get("TRANSFORMERS_RUNTIME_AUDIT", {}))

for result in hive_results:
    idx = int(result.get("index") or (len(manifest["outputs"]) + 1))
    if "article" in result:
        meta = write_article_bundle(result, idx)
        manifest["outputs"].append(meta)
        print(f"wrote {meta['path']} ({meta['bytes']} bytes, {meta['citation_count']} citations, worker={meta.get('worker_id')})")
    else:
        err = {
            "index": idx,
            "topic": result.get("topic"),
            "worker_id": result.get("worker_id"),
            "error": result.get("error", "unknown error"),
        }
        manifest["outputs"].append(err)
        print("ERROR:", err)

manifest["elapsed_seconds"] = round(time.time() - start_time, 2)
manifest_path = OUTPUT_DIR / "manifest.json"
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")

index_lines = [
    f"# {BRAND_NAME} Gemma 4 E2B Hive Blog Batch",
    "",
    f"Generated articles: {len([x for x in manifest['outputs'] if 'path' in x])}",
    f"Requested workers: {GEMMA_THREAD_COUNT}",
    f"Resolved workers: {hive_meta.get('resolved_threads')}",
    "",
]
for item in manifest["outputs"]:
    if "path" in item:
        rel = Path(item["path"]).name
        index_lines.append(f"- [{rel}](./{rel}) — {item.get('topic','')} — worker `{item.get('worker_id')}`")
    else:
        index_lines.append(f"- ERROR article {item.get('index')}: {item.get('error')}")
(OUTPUT_DIR / "index.md").write_text("\n".join(index_lines), encoding="utf-8")

zip_base = str(OUTPUT_DIR)
zip_path = shutil.make_archive(zip_base, "zip", OUTPUT_DIR)
print("\nDone.")
print("Manifest:", manifest_path)
print("ZIP:", zip_path)
print("Hive stats:", json.dumps(hive_meta, ensure_ascii=False, indent=2)[:4000])
print("Open the Files panel in Colab, or run the next cell for download links.")


In [ ]:
# Cell 8 — Colab download helpers

from pathlib import Path

try:
    from google.colab import files
    print("Downloading ZIP bundle...")
    files.download(str(Path(str(OUTPUT_DIR) + ".zip")))
except Exception as exc:
    print("Colab download helper unavailable. Files are here:")
    print(OUTPUT_DIR)
    print(Path(str(OUTPUT_DIR) + ".zip"))
    print("Reason:", exc)
